In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
import glob

# The correct path you found
dataset_path = '/kaggle/input/datasets/dravvalt/drishti-dataset/'

# Search for both .mp4 and .mkv files
mp4_files = glob.glob(dataset_path + '**/*.mp4', recursive=True)
mkv_files = glob.glob(dataset_path + '**/*.mkv', recursive=True)

# Combine them into one list
video_files = mp4_files + mkv_files

print(f"Found {len(video_files)} videos.")

# Print the first few to verify
for video in video_files[:10]:
    print(video)

In [ ]:
import cv2
import pandas as pd
import subprocess
import json
import os
from pathlib import Path

def run_ffprobe(video_path):
    """
    Read container/video-stream metadata.
    Does NOT decode the whole video.
    """
    cmd = [
        "ffprobe",
        "-v", "error",
        "-select_streams", "v:0",
        "-show_entries",
        "stream=codec_name,width,height,r_frame_rate,avg_frame_rate,pix_fmt",
        "-show_entries",
        "format=format_name,duration,size,bit_rate",
        "-of", "json",
        video_path
    ]

    try:
        result = subprocess.run(
            cmd,
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            text=True,
            timeout=60
        )

        if result.returncode != 0:
            return None, result.stderr

        return json.loads(result.stdout), None

    except Exception as e:
        return None, str(e)


def fraction_to_float(value):
    """
    Converts values such as '25/1' or '30000/1001' to float.
    """
    if not value:
        return None

    try:
        if "/" in value:
            num, den = value.split("/")
            den = float(den)

            if den == 0:
                return None

            return float(num) / den

        return float(value)

    except:
        return None


records = []

for i, video_path in enumerate(video_files, 1):

    print("=" * 90)
    print(f"[{i}/{len(video_files)}] {Path(video_path).name}")

    file_size_bytes = os.path.getsize(video_path)
    file_size_gb = file_size_bytes / (1024 ** 3)

    # ----------------------------------------------------
    # OpenCV metadata
    # ----------------------------------------------------

    cap = cv2.VideoCapture(video_path)

    opencv_opened = cap.isOpened()

    if opencv_opened:

        cv_fps = cap.get(cv2.CAP_PROP_FPS)
        cv_frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

        cv_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        cv_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

        if cv_fps and cv_fps > 0:
            cv_duration = cv_frame_count / cv_fps
        else:
            cv_duration = None

    else:
        cv_fps = None
        cv_frame_count = None
        cv_width = None
        cv_height = None
        cv_duration = None

    cap.release()

    # ----------------------------------------------------
    # FFprobe metadata
    # ----------------------------------------------------

    probe, probe_error = run_ffprobe(video_path)

    codec = None
    pixel_format = None
    container = None

    ff_duration = None
    ff_bitrate = None
    ff_avg_fps = None
    ff_nominal_fps = None

    if probe:

        streams = probe.get("streams", [])
        fmt = probe.get("format", {})

        if streams:
            stream = streams[0]

            codec = stream.get("codec_name")
            pixel_format = stream.get("pix_fmt")

            ff_avg_fps = fraction_to_float(
                stream.get("avg_frame_rate")
            )

            ff_nominal_fps = fraction_to_float(
                stream.get("r_frame_rate")
            )

        container = fmt.get("format_name")

        try:
            ff_duration = float(fmt.get("duration"))
        except:
            ff_duration = None

        try:
            ff_bitrate = int(fmt.get("bit_rate"))
        except:
            ff_bitrate = None

    # ----------------------------------------------------
    # Final preferred values
    # ----------------------------------------------------

    final_duration = (
        ff_duration
        if ff_duration is not None
        else cv_duration
    )

    final_fps = (
        ff_avg_fps
        if ff_avg_fps is not None and ff_avg_fps > 0
        else cv_fps
    )

    # ----------------------------------------------------
    # Record
    # ----------------------------------------------------

    record = {

        "filename": Path(video_path).name,
        "path": video_path,

        "extension": Path(video_path).suffix.lower(),

        "size_gb": round(file_size_gb, 4),

        "opencv_opened": opencv_opened,

        "container": container,
        "codec": codec,
        "pixel_format": pixel_format,

        "width": cv_width,
        "height": cv_height,

        "fps": round(final_fps, 3)
        if final_fps is not None else None,

        "nominal_fps": round(ff_nominal_fps, 3)
        if ff_nominal_fps is not None else None,

        "frame_count": cv_frame_count,

        "duration_sec": round(final_duration, 2)
        if final_duration is not None else None,

        "duration_min": round(final_duration / 60, 2)
        if final_duration is not None else None,

        "duration_hr": round(final_duration / 3600, 3)
        if final_duration is not None else None,

        "bitrate_mbps": round(ff_bitrate / 1_000_000, 3)
        if ff_bitrate is not None else None,

        "ffprobe_error": probe_error
    }

    records.append(record)

    print(f"Size       : {record['size_gb']} GB")
    print(f"Container  : {container}")
    print(f"Codec      : {codec}")
    print(f"Resolution : {cv_width} x {cv_height}")
    print(f"FPS        : {record['fps']}")
    print(f"Frames     : {cv_frame_count}")
    print(f"Duration   : {record['duration_min']} min")
    print(f"OpenCV     : {opencv_opened}")


audit_df = pd.DataFrame(records)

print("\n\nAUDIT COMPLETE")
display(audit_df)

In [ ]:
print("=" * 90)
print("DRISHTI DATASET — FORENSIC SUMMARY")
print("=" * 90)

print(f"\nTotal videos       : {len(audit_df)}")
print(f"Total size         : {audit_df['size_gb'].sum():.2f} GB")
print(f"Total duration     : {audit_df['duration_min'].sum():.2f} minutes")
print(f"Total duration     : {audit_df['duration_hr'].sum():.2f} hours")

print("\n" + "-" * 90)
print("FORMATS")
print("-" * 90)
print(audit_df["extension"].value_counts(dropna=False))

print("\n" + "-" * 90)
print("CODECS")
print("-" * 90)
print(audit_df["codec"].value_counts(dropna=False))

print("\n" + "-" * 90)
print("RESOLUTIONS")
print("-" * 90)

resolution_counts = (
    audit_df
    .groupby(["width", "height"])
    .size()
    .sort_values(ascending=False)
)

print(resolution_counts)

print("\n" + "-" * 90)
print("FPS DISTRIBUTION")
print("-" * 90)

print(
    audit_df["fps"]
    .value_counts(dropna=False)
    .sort_index()
)

print("\n" + "-" * 90)
print("VIDEOS — LARGEST FIRST")
print("-" * 90)

display(
    audit_df[
        [
            "filename",
            "size_gb",
            "duration_min",
            "width",
            "height",
            "fps",
            "frame_count",
            "codec"
        ]
    ]
    .sort_values("size_gb", ascending=False)
    .reset_index(drop=True)
)

In [ ]:
from pathlib import Path

AUDIT_OUTPUT = Path("/kaggle/working/drishti_audit")
AUDIT_OUTPUT.mkdir(parents=True, exist_ok=True)

csv_path = AUDIT_OUTPUT / "dataset_manifest.csv"
json_path = AUDIT_OUTPUT / "dataset_manifest.json"

audit_df.to_csv(
    csv_path,
    index=False
)

audit_df.to_json(
    json_path,
    orient="records",
    indent=2
)

print("Saved:")
print(csv_path)
print(json_path)

In [ ]:
import cv2
import numpy as np
from pathlib import Path

decode_results = []

for i, row in audit_df.iterrows():

    video_path = row["path"]
    duration_sec = row["duration_sec"]

    print("=" * 90)
    print(Path(video_path).name)

    cap = cv2.VideoCapture(video_path)

    if not cap.isOpened():

        decode_results.append({
            "filename": Path(video_path).name,
            "first_frame": False,
            "middle_frame": False,
            "last_frame": False
        })

        print("FAILED TO OPEN")
        continue

    if duration_sec is None or duration_sec <= 1:

        timestamps = [
            0.0,
            0.5,
            1.0
        ]

    else:

        timestamps = [
            0.0,
            duration_sec / 2,
            max(0, duration_sec - 1.0)
        ]

    statuses = []

    labels = [
        "first",
        "middle",
        "last"
    ]

    for label, timestamp in zip(labels, timestamps):

        cap.set(
            cv2.CAP_PROP_POS_MSEC,
            timestamp * 1000
        )

        ok, frame = cap.read()

        valid = (
            ok and
            frame is not None and
            frame.size > 0
        )

        statuses.append(valid)

        print(
            f"{label:7s} "
            f"@ {timestamp:10.2f}s "
            f"→ {'PASS' if valid else 'FAIL'}"
        )

    cap.release()

    decode_results.append({
        "filename": Path(video_path).name,
        "first_frame": statuses[0],
        "middle_frame": statuses[1],
        "last_frame": statuses[2]
    })


decode_df = pd.DataFrame(decode_results)

display(decode_df)

In [ ]:
decode_df["all_decode_checks_pass"] = (
    decode_df["first_frame"] &
    decode_df["middle_frame"] &
    decode_df["last_frame"]
)

print("Decode test summary:")
print(
    decode_df["all_decode_checks_pass"]
    .value_counts()
)

problem_videos = decode_df[
    ~decode_df["all_decode_checks_pass"]
]

if len(problem_videos) == 0:

    print("\n✅ All videos passed first/middle/last frame decoding.")

else:

    print("\n⚠️ Videos requiring attention:")

    display(problem_videos)

In [ ]:
sampling_rates = [
    1,
    2,
    5,
    10
]

workload_rows = []

for _, row in audit_df.iterrows():

    duration = row["duration_sec"]

    if duration is None:
        continue

    result = {

        "filename": row["filename"],

        "duration_min": row["duration_min"],

        "native_fps": row["fps"],

        "native_frames": row["frame_count"]
    }

    for rate in sampling_rates:

        estimated_frames = int(
            duration * rate
        )

        result[f"frames_at_{rate}fps"] = estimated_frames

    workload_rows.append(result)


workload_df = pd.DataFrame(workload_rows)

display(workload_df)

In [ ]:
print("=" * 90)
print("TOTAL MULTI-HMR INFERENCE WORKLOAD")
print("=" * 90)

for rate in sampling_rates:

    total_frames = workload_df[
        f"frames_at_{rate}fps"
    ].sum()

    print(
        f"{rate:2d} FPS → "
        f"{total_frames:,} inference frames"
    )

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import math

# ---------------------------------------------------------
# OUTPUT DIRECTORY
# ---------------------------------------------------------

CONTACT_DIR = Path("/kaggle/working/drishti_audit/contact_sheets")
CONTACT_DIR.mkdir(parents=True, exist_ok=True)

# Number of representative frames per video
N_SAMPLES = 9


# ---------------------------------------------------------
# HELPER: READ FRAME AT TIMESTAMP
# ---------------------------------------------------------

def read_frame_at_time(video_path, timestamp_sec):
    """
    Seek directly to timestamp without decoding the entire video.
    Returns RGB frame or None.
    """

    cap = cv2.VideoCapture(str(video_path))

    if not cap.isOpened():
        return None

    cap.set(
        cv2.CAP_PROP_POS_MSEC,
        timestamp_sec * 1000
    )

    success, frame = cap.read()

    cap.release()

    if not success or frame is None:
        return None

    # OpenCV BGR -> matplotlib RGB
    frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    return frame


# ---------------------------------------------------------
# GENERATE CONTACT SHEETS
# ---------------------------------------------------------

contact_sheet_records = []

for video_idx, row in audit_df.iterrows():

    video_path = row["path"]
    video_name = row["filename"]
    duration = row["duration_sec"]

    print("=" * 100)
    print(f"[{video_idx + 1}/{len(audit_df)}]")
    print(video_name)

    if duration is None or duration <= 0:
        print("⚠️ Invalid duration — skipped.")
        continue

    # Avoid exact first/last frame.
    # Sample from 5% -> 95%.
    timestamps = np.linspace(
        duration * 0.05,
        duration * 0.95,
        N_SAMPLES
    )

    frames = []
    successful_timestamps = []

    for timestamp in timestamps:

        frame = read_frame_at_time(
            video_path,
            float(timestamp)
        )

        if frame is not None:

            frames.append(frame)
            successful_timestamps.append(timestamp)

            print(
                f"✓ {timestamp:8.2f}s "
                f"({timestamp/60:6.2f} min)"
            )

        else:

            frames.append(None)
            successful_timestamps.append(timestamp)

            print(
                f"✗ FAILED @ {timestamp:.2f}s"
            )

    # -----------------------------------------------------
    # CONTACT SHEET
    # -----------------------------------------------------

    fig, axes = plt.subplots(
        3,
        3,
        figsize=(18, 11)
    )

    axes = axes.flatten()

    for i, ax in enumerate(axes):

        ax.axis("off")

        timestamp = successful_timestamps[i]
        frame = frames[i]

        if frame is not None:

            ax.imshow(frame)

            ax.set_title(
                f"{timestamp/60:.2f} min\n"
                f"({timestamp:.1f}s)",
                fontsize=10
            )

        else:

            ax.text(
                0.5,
                0.5,
                "FRAME DECODE FAILED",
                horizontalalignment="center",
                verticalalignment="center",
                fontsize=12
            )

    fig.suptitle(
        video_name,
        fontsize=14,
        fontweight="bold"
    )

    plt.tight_layout()

    # Safe filename
    safe_name = (
        Path(video_name).stem
        .replace(" ", "_")
        .replace("/", "_")
        .replace("\\", "_")
    )

    output_path = (
        CONTACT_DIR /
        f"{video_idx:02d}_{safe_name}_contact_sheet.jpg"
    )

    plt.savefig(
        output_path,
        dpi=130,
        bbox_inches="tight"
    )

    plt.show()
    plt.close(fig)

    contact_sheet_records.append({
        "filename": video_name,
        "contact_sheet": str(output_path),
        "frames_requested": N_SAMPLES,
        "frames_decoded": sum(
            frame is not None
            for frame in frames
        )
    })

    print(f"\nSaved → {output_path}\n")


print("=" * 100)
print("CONTACT SHEET GENERATION COMPLETE")
print("=" * 100)
print(f"Saved to: {CONTACT_DIR}")

In [ ]:
import cv2
import pandas as pd
from pathlib import Path

# ---------------------------------------------------------
# SETTINGS
# ---------------------------------------------------------

WINDOW_SECONDS = 10.0

# Test beginning, quarter, middle, three-quarter, end-ish
WINDOW_POSITIONS = [
    0.05,
    0.25,
    0.50,
    0.75,
    0.90,
]

health_records = []


# ---------------------------------------------------------
# SEQUENTIAL WINDOW TEST
# ---------------------------------------------------------

for video_idx, row in audit_df.iterrows():

    video_path = row["path"]
    video_name = row["filename"]

    duration = row["duration_sec"]
    fps = row["fps"]

    print("=" * 100)
    print(f"[{video_idx + 1}/{len(audit_df)}] {video_name}")

    if duration is None or fps is None or fps <= 0:

        print("⚠️ Missing duration/FPS — skipped.")

        health_records.append({
            "filename": video_name,
            "windows_tested": 0,
            "frames_expected": 0,
            "frames_decoded": 0,
            "decode_failures": None,
            "decode_success_rate": None
        })

        continue

    total_expected = 0
    total_decoded = 0
    total_failures = 0

    for position in WINDOW_POSITIONS:

        start_sec = duration * position

        # Ensure the 10-second window stays inside video
        start_sec = min(
            start_sec,
            max(0, duration - WINDOW_SECONDS - 1)
        )

        cap = cv2.VideoCapture(video_path)

        if not cap.isOpened():

            print(
                f"✗ Could not open window "
                f"starting at {start_sec:.2f}s"
            )

            total_failures += int(
                WINDOW_SECONDS * fps
            )

            continue

        # Seek ONCE
        cap.set(
            cv2.CAP_PROP_POS_MSEC,
            start_sec * 1000
        )

        target_frames = max(
            1,
            int(round(WINDOW_SECONDS * fps))
        )

        decoded = 0
        failures = 0

        # Then decode SEQUENTIALLY
        for _ in range(target_frames):

            ok, frame = cap.read()

            if (
                ok
                and frame is not None
                and frame.size > 0
            ):
                decoded += 1

            else:
                failures += 1

        cap.release()

        total_expected += target_frames
        total_decoded += decoded
        total_failures += failures

        success_rate = (
            decoded / target_frames
            if target_frames > 0
            else 0
        )

        print(
            f"{start_sec:9.2f}s "
            f"→ {decoded:4d}/{target_frames:4d} frames "
            f"({100 * success_rate:6.2f}%)"
        )

    overall_rate = (
        total_decoded / total_expected
        if total_expected > 0
        else 0
    )

    health_records.append({
        "filename": video_name,
        "windows_tested": len(WINDOW_POSITIONS),
        "frames_expected": total_expected,
        "frames_decoded": total_decoded,
        "decode_failures": total_failures,
        "decode_success_rate": round(
            overall_rate,
            6
        )
    })

    print(
        f"\nOverall sequential decode: "
        f"{total_decoded}/{total_expected} "
        f"({100 * overall_rate:.3f}%)"
    )


# ---------------------------------------------------------
# RESULTS
# ---------------------------------------------------------

sequential_health_df = pd.DataFrame(
    health_records
)

display(sequential_health_df)

In [ ]:
sequential_health_df["status"] = (
    sequential_health_df["decode_success_rate"]
    .apply(
        lambda x:
            "PASS"
            if x is not None and x >= 0.995
            else "REVIEW"
    )
)

display(
    sequential_health_df[
        [
            "filename",
            "frames_expected",
            "frames_decoded",
            "decode_failures",
            "decode_success_rate",
            "status"
        ]
    ]
)

print("\nStatus counts:")
print(
    sequential_health_df["status"]
    .value_counts()
)

In [ ]:
import cv2
import time
from pathlib import Path

# ---------------------------------------------------------
# FIND PAPER VIDEO
# ---------------------------------------------------------

paper_candidates = [
    p for p in video_files
    if "piece of paper" in Path(p).name.lower()
]

assert len(paper_candidates) == 1, (
    f"Expected exactly 1 paper video, found {len(paper_candidates)}"
)

paper_video = paper_candidates[0]

print("Testing:")
print(Path(paper_video).name)
print()


# ---------------------------------------------------------
# OPEN VIDEO
# ---------------------------------------------------------

cap = cv2.VideoCapture(paper_video)

assert cap.isOpened(), "Could not open paper video."

reported_fps = cap.get(cv2.CAP_PROP_FPS)
reported_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
reported_duration = (
    reported_frames / reported_fps
    if reported_fps > 0
    else None
)

print(f"Reported FPS        : {reported_fps}")
print(f"Reported frames     : {reported_frames}")
print(f"Reported duration   : {reported_duration:.3f} sec")
print()


# ---------------------------------------------------------
# FULL SEQUENTIAL DECODE
# ---------------------------------------------------------

decoded_frames = 0
first_timestamp_ms = None
last_timestamp_ms = None

start_time = time.perf_counter()

while True:

    ok, frame = cap.read()

    if not ok or frame is None:
        break

    timestamp_ms = cap.get(cv2.CAP_PROP_POS_MSEC)

    if first_timestamp_ms is None:
        first_timestamp_ms = timestamp_ms

    last_timestamp_ms = timestamp_ms

    decoded_frames += 1


elapsed = time.perf_counter() - start_time

cap.release()


# ---------------------------------------------------------
# REPORT
# ---------------------------------------------------------

missing_vs_metadata = reported_frames - decoded_frames

print("=" * 80)
print("FULL SEQUENTIAL DECODE RESULT")
print("=" * 80)

print(f"Reported frames     : {reported_frames}")
print(f"Actually decoded    : {decoded_frames}")
print(f"Difference          : {missing_vs_metadata}")

if first_timestamp_ms is not None:
    print(
        f"First timestamp     : "
        f"{first_timestamp_ms / 1000:.3f} sec"
    )

if last_timestamp_ms is not None:
    print(
        f"Last timestamp      : "
        f"{last_timestamp_ms / 1000:.3f} sec"
    )

    print(
        f"Decoded timeline    : "
        f"{last_timestamp_ms / 1000 / 60:.3f} min"
    )

print(f"Decode wall time    : {elapsed:.2f} sec")

if elapsed > 0:
    print(
        f"Decode throughput   : "
        f"{decoded_frames / elapsed:.2f} frames/sec"
    )


# ---------------------------------------------------------
# VERDICT
# ---------------------------------------------------------

ratio = (
    decoded_frames / reported_frames
    if reported_frames > 0
    else 0
)

print()
print(f"Frame coverage      : {ratio * 100:.3f}%")

if ratio >= 0.995:
    print("✅ FULL SEQUENTIAL INTEGRITY: PASS")

elif ratio >= 0.98:
    print("⚠️ FULL SEQUENTIAL INTEGRITY: MINOR MISMATCH")

else:
    print("❌ FULL SEQUENTIAL INTEGRITY: REVIEW REQUIRED")

In [ ]:
import subprocess
import json
from pathlib import Path

paper_video = [
    p for p in video_files
    if "piece of paper" in Path(p).name.lower()
][0]

print("VIDEO:")
print(Path(paper_video).name)
print()


# ---------------------------------------------------------
# FFPROBE: COUNT ACTUAL READABLE FRAMES + PACKETS
# ---------------------------------------------------------

cmd = [
    "ffprobe",
    "-v", "error",

    "-select_streams", "v:0",

    "-count_frames",
    "-count_packets",

    "-show_entries",
    (
        "stream="
        "codec_name,"
        "r_frame_rate,"
        "avg_frame_rate,"
        "duration,"
        "nb_frames,"
        "nb_read_frames,"
        "nb_read_packets"
    ),

    "-show_entries",
    "format=duration",

    "-of", "json",

    paper_video
]

result = subprocess.run(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True,
    timeout=120
)

print("FFPROBE RETURN CODE:", result.returncode)

if result.stderr.strip():
    print("\nFFPROBE STDERR:")
    print(result.stderr)

probe = json.loads(result.stdout)

print("\nRAW FFPROBE RESULT:")
print(
    json.dumps(
        probe,
        indent=2
    )
)


# ---------------------------------------------------------
# CLEAN SUMMARY
# ---------------------------------------------------------

stream = probe["streams"][0]

print("\n" + "=" * 80)
print("FRAME / PACKET DIAGNOSTIC")
print("=" * 80)

print("Codec             :", stream.get("codec_name"))
print("r_frame_rate      :", stream.get("r_frame_rate"))
print("avg_frame_rate    :", stream.get("avg_frame_rate"))

print("Metadata nb_frames:", stream.get("nb_frames"))

print(
    "Actually readable frames:",
    stream.get("nb_read_frames")
)

print(
    "Readable packets       :",
    stream.get("nb_read_packets")
)

print(
    "Stream duration        :",
    stream.get("duration")
)

print(
    "Container duration     :",
    probe.get("format", {}).get("duration")
)

In [ ]:
import cv2
import numpy as np
import pandas as pd
from pathlib import Path

paper_video = [
    p for p in video_files
    if "piece of paper" in Path(p).name.lower()
][0]

print("TIMESTAMP AUDIT")
print(Path(paper_video).name)
print("=" * 90)

cap = cv2.VideoCapture(paper_video)

assert cap.isOpened(), "Could not open video"

timestamps = []
frame_indices = []

frame_idx = 0

while True:

    ok, frame = cap.read()

    if not ok or frame is None:
        break

    timestamp_sec = (
        cap.get(cv2.CAP_PROP_POS_MSEC) / 1000.0
    )

    timestamps.append(timestamp_sec)
    frame_indices.append(frame_idx)

    frame_idx += 1

cap.release()

timestamps = np.array(timestamps, dtype=float)

print(f"Decoded frames : {len(timestamps)}")
print(f"First timestamp: {timestamps[0]:.6f} sec")
print(f"Last timestamp : {timestamps[-1]:.6f} sec")


# ---------------------------------------------------------
# TIMESTAMP DIFFERENCES
# ---------------------------------------------------------

dt = np.diff(timestamps)

print("\n" + "=" * 90)
print("TIMESTAMP DELTA STATISTICS")
print("=" * 90)

print(f"Min dt    : {dt.min():.6f} sec")
print(f"Median dt : {np.median(dt):.6f} sec")
print(f"Mean dt   : {dt.mean():.6f} sec")
print(f"P95 dt    : {np.percentile(dt, 95):.6f} sec")
print(f"Max dt    : {dt.max():.6f} sec")


# ---------------------------------------------------------
# INTEGRITY CHECKS
# ---------------------------------------------------------

backward = np.where(dt < 0)[0]
duplicate = np.where(dt == 0)[0]

# Anything over 2x median cadence is worth inspecting
median_dt = np.median(dt)

large_gap_threshold = max(
    0.1,
    2 * median_dt
)

large_gaps = np.where(
    dt > large_gap_threshold
)[0]


print("\n" + "=" * 90)
print("INTEGRITY CHECK")
print("=" * 90)

print("Backward timestamps :", len(backward))
print("Duplicate timestamps:", len(duplicate))

print(
    f"Large gaps > {large_gap_threshold:.4f}s:",
    len(large_gaps)
)


# ---------------------------------------------------------
# SHOW LARGEST GAPS
# ---------------------------------------------------------

gap_df = pd.DataFrame({
    "frame_before": np.arange(len(dt)),
    "timestamp_before": timestamps[:-1],
    "timestamp_after": timestamps[1:],
    "dt_sec": dt
})

largest_gaps = (
    gap_df
    .sort_values("dt_sec", ascending=False)
    .head(20)
    .reset_index(drop=True)
)

print("\nLargest timestamp gaps:")

display(largest_gaps)


# ---------------------------------------------------------
# VERDICT
# ---------------------------------------------------------

print("\n" + "=" * 90)
print("TIMESTAMP VERDICT")
print("=" * 90)

if len(backward) > 0:

    print("❌ FAIL: timestamps move backwards.")

elif len(duplicate) > 0:

    print(
        "⚠️ REVIEW: duplicate timestamps exist."
    )

elif dt.max() > 1.0:

    print(
        "⚠️ REVIEW: at least one >1 second temporal gap exists."
    )

else:

    print(
        "✅ PASS: timeline is monotonic and usable."
    )

In [ ]:
# CELL 13 — VERIFIED SCENE FEASIBILITY MANIFEST
# Visual scene properties only.
# IMPORTANT:
# filename_scenario_hint is NOT ground-truth behaviour annotation.

import pandas as pd

scene_data = [
    {
        "video_id": "VIDEO_01",
        "filename": "06.Candidate was found using a mobile phone in the examination hall..mp4",

        "scene_type": "CBT_EXAM_LAB",
        "people_visibility": "MULTIPLE",
        "seating_fixed": "YES",

        "board_visible": "NO",
        "own_monitor_context": "YES",
        "aisle_visible": "YES",
        "invigilator_context": "POSSIBLE_NOT_YET_CALIBRATED",

        "head_visibility": "GOOD",
        "torso_visibility": "GOOD",
        "hand_visibility": "MODERATE",

        "occlusion_level": "MEDIUM",
        "camera_angle": "HIGH_OBLIQUE",

        "smplx_feasibility": "MEDIUM_HIGH",

        "filename_scenario_hint": "MOBILE_PHONE_USAGE",

        "notes": (
            "Long CBT-lab recording. Cubicle partitions cause partial lower-body "
            "and hand occlusion. Suitable later for long-duration temporal testing."
        )
    },

    {
        "video_id": "VIDEO_02",
        "filename": "During the exam, the candidates exchanged their seats with each other..mp4",

        "scene_type": "CBT_EXAM_LAB",
        "people_visibility": "MULTIPLE",
        "seating_fixed": "YES",

        "board_visible": "NO",
        "own_monitor_context": "YES",
        "aisle_visible": "YES",
        "invigilator_context": "POSSIBLE_NOT_YET_CALIBRATED",

        "head_visibility": "GOOD",
        "torso_visibility": "GOOD",
        "hand_visibility": "MODERATE",

        "occlusion_level": "MEDIUM",
        "camera_angle": "HIGH_OBLIQUE",

        "smplx_feasibility": "MEDIUM_HIGH",

        "filename_scenario_hint": "SEAT_EXCHANGE",

        "notes": (
            "Useful later for person-track versus seat-identity testing. "
            "Sparse contact sheets do not establish the exact seat-exchange interval."
        )
    },

    {
        "video_id": "VIDEO_03",
        "filename": "05.Crowd observed near the reception and verification desk..mp4",

        "scene_type": "RECEPTION_LOBBY",
        "people_visibility": "MULTIPLE_MOVING",
        "seating_fixed": "NO",

        "board_visible": "NO",
        "own_monitor_context": "NO",
        "aisle_visible": "NOT_APPLICABLE",
        "invigilator_context": "NOT_APPLICABLE",

        "head_visibility": "GOOD",
        "torso_visibility": "GOOD",
        "hand_visibility": "MODERATE",

        "occlusion_level": "HIGH",
        "camera_angle": "HIGH_OBLIQUE",

        "smplx_feasibility": "LOW_MEDIUM",

        "filename_scenario_hint": "CROWD_GATHERING",

        "notes": (
            "Not a seated exam-hall scene. Useful for crowd/occlusion/association "
            "robustness, not the first behavioural calibration clip."
        )
    },

    {
        "video_id": "VIDEO_04",
        "filename": "01.Candidate was found using a mobile phone in the examination hall..mkv",

        "scene_type": "CBT_EXAM_LAB",
        "people_visibility": "FEW",
        "seating_fixed": "YES",

        "board_visible": "NO",
        "own_monitor_context": "YES",
        "aisle_visible": "YES",
        "invigilator_context": "POSSIBLE_NOT_YET_CALIBRATED",

        "head_visibility": "GOOD",
        "torso_visibility": "MODERATE",
        "hand_visibility": "MODERATE",

        "occlusion_level": "MEDIUM",
        "camera_angle": "HIGH_OBLIQUE",

        "smplx_feasibility": "MEDIUM",

        "filename_scenario_hint": "MOBILE_PHONE_USAGE",

        "notes": (
            "Upper body visible but workstation/cubicle geometry partially "
            "occludes hands and lower body."
        )
    },

    {
        "video_id": "VIDEO_05",
        "filename": "03.CCTV Mobile Usage.mkv",

        "scene_type": "CBT_EXAM_LAB",
        "people_visibility": "PRIMARY_SINGLE_SUBJECT",
        "seating_fixed": "YES",

        "board_visible": "NO",
        "own_monitor_context": "YES",
        "aisle_visible": "YES",
        "invigilator_context": "POSSIBLE_NOT_YET_CALIBRATED",

        "head_visibility": "GOOD",
        "torso_visibility": "GOOD",
        "hand_visibility": "MODERATE",

        "occlusion_level": "LOW_MEDIUM",
        "camera_angle": "HIGH_OBLIQUE",

        "smplx_feasibility": "HIGH",

        "filename_scenario_hint": "MOBILE_PHONE_USAGE",

        "notes": (
            "Best initial Multi-HMR/SMPL-X feasibility candidate: one dominant "
            "seated subject, 1280x720, manageable 4.70-minute duration. "
            "Native ~12 FPS is lower than other clips but still suitable for "
            "timestamp-based temporal analysis."
        )
    },

    {
        "video_id": "VIDEO_06",
        "filename": "02.Candidate was found using a mobile phone in the examination hall..mkv",

        "scene_type": "CBT_EXAM_LAB",
        "people_visibility": "FEW",
        "seating_fixed": "YES",

        "board_visible": "NO",
        "own_monitor_context": "YES",
        "aisle_visible": "YES",
        "invigilator_context": "POSSIBLE_NOT_YET_CALIBRATED",

        "head_visibility": "GOOD",
        "torso_visibility": "MODERATE",
        "hand_visibility": "MODERATE",

        "occlusion_level": "MEDIUM",
        "camera_angle": "HIGH_OBLIQUE",

        "smplx_feasibility": "MEDIUM",

        "filename_scenario_hint": "MOBILE_PHONE_USAGE",

        "notes": (
            "Useful secondary phone-use scene after initial feasibility. "
            "Several foreground/background people appear across sampled frames."
        )
    },

    {
        "video_id": "VIDEO_07",
        "filename": "Seat No. 12 was seen taking a piece of paper from the desk.mkv",

        "scene_type": "CBT_EXAM_LAB",
        "people_visibility": "MULTIPLE",
        "seating_fixed": "YES",

        "board_visible": "NO",
        "own_monitor_context": "YES",
        "aisle_visible": "YES",
        "invigilator_context": "POSSIBLE_NOT_YET_CALIBRATED",

        "head_visibility": "GOOD",
        "torso_visibility": "GOOD",
        "hand_visibility": "MODERATE",

        "occlusion_level": "MEDIUM_HIGH",
        "camera_angle": "HIGH_OBLIQUE",

        "smplx_feasibility": "MEDIUM_HIGH",

        "filename_scenario_hint": "PAPER_INTERACTION",

        "notes": (
            "Actual stream contains 1936 readable packets/frames and reaches "
            "~88.36 s of an ~88.40 s timeline. Timestamps are monotonic. "
            "Decoder emits MPEG-4 header/keyframe warnings, but do NOT describe "
            "this as 274 confirmed missing encoded frames."
        )
    },

    {
        "video_id": "VIDEO_08",
        "filename": "04.CCTV Candidate Talking.mkv",

        "scene_type": "CBT_EXAM_LAB",
        "people_visibility": "MULTIPLE",
        "seating_fixed": "YES",

        "board_visible": "NO",
        "own_monitor_context": "YES",
        "aisle_visible": "LIMITED",
        "invigilator_context": "POSSIBLE_NOT_YET_CALIBRATED",

        "head_visibility": "MODERATE",
        "torso_visibility": "MODERATE",
        "hand_visibility": "LOW_MODERATE",

        "occlusion_level": "MEDIUM",
        "camera_angle": "HIGH_OBLIQUE",

        "smplx_feasibility": "LOW_MEDIUM",

        "filename_scenario_hint": "POSSIBLE_CANDIDATE_TALKING",

        "notes": (
            "640x480 at 8 FPS. Useful later for low-resolution robustness "
            "and reciprocal head/torso interaction experiments. Exact talking "
            "interval must still be manually annotated."
        )
    }
]

scene_audit_df = pd.DataFrame(scene_data)

# ---------------------------------------------------------
# MERGE VERIFIED TECHNICAL METADATA
# ---------------------------------------------------------

metadata = audit_df[
    [
        "filename",
        "width",
        "height",
        "fps",
        "duration_min",
        "codec"
    ]
].copy()

metadata["resolution"] = (
    metadata["width"].astype(int).astype(str)
    + "x"
    + metadata["height"].astype(int).astype(str)
)

scene_audit_df = scene_audit_df.merge(
    metadata[
        [
            "filename",
            "resolution",
            "fps",
            "duration_min",
            "codec"
        ]
    ],
    on="filename",
    how="left"
)


# ---------------------------------------------------------
# ORDER
# ---------------------------------------------------------

ordered_cols = [
    "video_id",
    "filename",
    "resolution",
    "fps",
    "duration_min",
    "codec",

    "scene_type",
    "people_visibility",
    "seating_fixed",

    "board_visible",
    "own_monitor_context",
    "aisle_visible",
    "invigilator_context",

    "head_visibility",
    "torso_visibility",
    "hand_visibility",

    "occlusion_level",
    "camera_angle",

    "smplx_feasibility",

    "filename_scenario_hint",

    "notes"
]

scene_audit_df = scene_audit_df[ordered_cols]


# ---------------------------------------------------------
# SAVE
# ---------------------------------------------------------

scene_manifest_path = (
    "/kaggle/working/drishti_audit/"
    "scene_audit_manifest.csv"
)

scene_audit_df.to_csv(
    scene_manifest_path,
    index=False
)

display(scene_audit_df)

print("\nSaved:")
print(scene_manifest_path)

In [ ]:
from PIL import Image
from IPython.display import display
from pathlib import Path

contact_sheet_paths = sorted(
    Path("/kaggle/working/drishti_audit/contact_sheets")
    .glob("*.jpg")
)

print(f"Found {len(contact_sheet_paths)} contact sheets.")

for i, path in enumerate(contact_sheet_paths):

    print("\n" + "=" * 100)
    print(f"[{i+1}/{len(contact_sheet_paths)}]")
    print(path.name)
    print("=" * 100)

    img = Image.open(path)

    display(img)

In [ ]:
# CELL 15 — Zip all contact sheets for easy download

from pathlib import Path
import shutil
from IPython.display import FileLink, display

CONTACT_DIR = Path("/kaggle/working/drishti_audit/contact_sheets")
ZIP_BASE = "/kaggle/working/drishti_contact_sheets"

# Creates:
# /kaggle/working/drishti_contact_sheets.zip
zip_path = shutil.make_archive(
    ZIP_BASE,
    "zip",
    root_dir=CONTACT_DIR
)

print(f"Created ZIP: {zip_path}")

print("\nFiles inside:")
for p in sorted(CONTACT_DIR.glob("*.jpg")):
    print(" -", p.name)

print("\nDownload:")
display(FileLink(zip_path))

In [ ]:
import cv2
import matplotlib.pyplot as plt
from pathlib import Path

# ---------------------------------------------------------
# SELECT FIRST MULTI-HMR FEASIBILITY VIDEO
# ---------------------------------------------------------

target_video = [
    p for p in video_files
    if Path(p).name == "03.CCTV Mobile Usage.mkv"
][0]

TARGET_TIMESTAMP = 140.91

print("Selected video:")
print(Path(target_video).name)

print(f"\nTimestamp: {TARGET_TIMESTAMP:.2f} sec")
print(f"Timestamp: {TARGET_TIMESTAMP/60:.2f} min")


# ---------------------------------------------------------
# EXTRACT FRAME
# ---------------------------------------------------------

cap = cv2.VideoCapture(target_video)

assert cap.isOpened(), "Could not open target video."

cap.set(
    cv2.CAP_PROP_POS_MSEC,
    TARGET_TIMESTAMP * 1000
)

ok, frame = cap.read()

actual_timestamp = (
    cap.get(cv2.CAP_PROP_POS_MSEC) / 1000.0
)

cap.release()

assert ok and frame is not None, "Failed to decode selected frame."


# ---------------------------------------------------------
# SAVE ORIGINAL FRAME
# ---------------------------------------------------------

frame_output = Path(
    "/kaggle/working/drishti_audit/"
    "first_multihmr_test_frame.jpg"
)

cv2.imwrite(
    str(frame_output),
    frame
)


# ---------------------------------------------------------
# DISPLAY
# ---------------------------------------------------------

frame_rgb = cv2.cvtColor(
    frame,
    cv2.COLOR_BGR2RGB
)

plt.figure(figsize=(14, 8))

plt.imshow(frame_rgb)

plt.title(
    f"FIRST MULTI-HMR TEST FRAME\n"
    f"{Path(target_video).name}\n"
    f"{actual_timestamp:.3f} sec"
)

plt.axis("off")

plt.show()


# ---------------------------------------------------------
# FRAME INFO
# ---------------------------------------------------------

height, width = frame.shape[:2]

print("=" * 70)
print("FRAME INFORMATION")
print("=" * 70)

print(f"Resolution       : {width} x {height}")
print(f"Requested time   : {TARGET_TIMESTAMP:.3f} sec")
print(f"Decoded time     : {actual_timestamp:.3f} sec")
print(f"Saved            : {frame_output}")

In [ ]:
# ============================================================
# DRISHTI — DRAVVYA PIPELINE
# SINGLE-VIDEO DEVELOPMENT TARGET
# VIDEO_05: 03.CCTV Mobile Usage.mkv
# ============================================================

import cv2
import json
from pathlib import Path
import matplotlib.pyplot as plt

# ------------------------------------------------------------
# 1. FREEZE TARGET VIDEO
# ------------------------------------------------------------

TARGET_VIDEO_NAME = "03.CCTV Mobile Usage.mkv"

matches = [
    p for p in video_files
    if Path(p).name == TARGET_VIDEO_NAME
]

assert len(matches) == 1, (
    f"Expected exactly one match for {TARGET_VIDEO_NAME}, "
    f"found {len(matches)}"
)

TARGET_VIDEO = matches[0]

print("=" * 90)
print("DRISHTI SINGLE-VIDEO DEVELOPMENT TARGET")
print("=" * 90)

print("Video:")
print(TARGET_VIDEO)


# ------------------------------------------------------------
# 2. READ VERIFIED METADATA FROM AUDIT
# ------------------------------------------------------------

target_meta = audit_df[
    audit_df["filename"] == TARGET_VIDEO_NAME
].iloc[0]

print("\nMetadata:")
print(f"Resolution : {int(target_meta['width'])}x{int(target_meta['height'])}")
print(f"FPS        : {target_meta['fps']}")
print(f"Duration   : {target_meta['duration_sec']:.2f} sec")
print(f"Duration   : {target_meta['duration_min']:.2f} min")
print(f"Codec      : {target_meta['codec']}")


# ------------------------------------------------------------
# 3. CREATE DEDICATED RUN DIRECTORY
# ------------------------------------------------------------

RUN_ROOT = Path(
    "/kaggle/working/drishti_video05_multihmr"
)

FRAME_DIR = RUN_ROOT / "frames"
OUTPUT_DIR = RUN_ROOT / "outputs"
CONFIG_DIR = RUN_ROOT / "config"

for directory in [
    RUN_ROOT,
    FRAME_DIR,
    OUTPUT_DIR,
    CONFIG_DIR
]:
    directory.mkdir(
        parents=True,
        exist_ok=True
    )

print("\nRun root:")
print(RUN_ROOT)


# ------------------------------------------------------------
# 4. FIRST FEASIBILITY TIMESTAMP
# ------------------------------------------------------------

# Chosen from our contact-sheet inspection.
TEST_TIMESTAMP_SEC = 140.91


# ------------------------------------------------------------
# 5. EXTRACT FRAME
# ------------------------------------------------------------

cap = cv2.VideoCapture(TARGET_VIDEO)

assert cap.isOpened(), "Could not open VIDEO_05"

cap.set(
    cv2.CAP_PROP_POS_MSEC,
    TEST_TIMESTAMP_SEC * 1000
)

ok, frame_bgr = cap.read()

actual_timestamp_sec = (
    cap.get(cv2.CAP_PROP_POS_MSEC) / 1000.0
)

cap.release()

assert ok and frame_bgr is not None, (
    "Failed to decode first Multi-HMR test frame"
)

height, width = frame_bgr.shape[:2]


# ------------------------------------------------------------
# 6. SAVE ORIGINAL FRAME
# ------------------------------------------------------------

frame_path = (
    FRAME_DIR /
    "video05_multihmr_feasibility_t140_91.jpg"
)

saved = cv2.imwrite(
    str(frame_path),
    frame_bgr
)

assert saved, "Failed to save test frame"


# ------------------------------------------------------------
# 7. DISPLAY LARGE
# ------------------------------------------------------------

frame_rgb = cv2.cvtColor(
    frame_bgr,
    cv2.COLOR_BGR2RGB
)

plt.figure(figsize=(16, 9))

plt.imshow(frame_rgb)

plt.title(
    "VIDEO_05 — FIRST MULTI-HMR / SMPL-X FEASIBILITY FRAME\n"
    f"Requested: {TEST_TIMESTAMP_SEC:.3f}s | "
    f"Decoded: {actual_timestamp_sec:.3f}s",
    fontsize=14
)

plt.axis("off")
plt.show()


# ------------------------------------------------------------
# 8. SAVE RUN MANIFEST
# ------------------------------------------------------------

manifest = {
    "video_id": "VIDEO_05",

    "filename": TARGET_VIDEO_NAME,

    "source_path": TARGET_VIDEO,

    "resolution": {
        "width": int(width),
        "height": int(height)
    },

    "video_fps_metadata": float(
        target_meta["fps"]
    ),

    "video_duration_sec": float(
        target_meta["duration_sec"]
    ),

    "test_frame": {
        "requested_timestamp_sec": TEST_TIMESTAMP_SEC,
        "decoded_timestamp_sec": float(
            actual_timestamp_sec
        ),
        "path": str(frame_path)
    },

    "development_stage": (
        "MULTI_HMR_SINGLE_FRAME_FEASIBILITY"
    )
}

manifest_path = (
    CONFIG_DIR /
    "video05_run_manifest.json"
)

with open(
    manifest_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        manifest,
        f,
        indent=2
    )


# ------------------------------------------------------------
# 9. FINAL REPORT
# ------------------------------------------------------------

print("\n" + "=" * 90)
print("VIDEO_05 INITIALIZATION COMPLETE")
print("=" * 90)

print(f"Frame resolution : {width}x{height}")
print(f"Requested time   : {TEST_TIMESTAMP_SEC:.3f}s")
print(f"Decoded time     : {actual_timestamp_sec:.3f}s")

print("\nSaved frame:")
print(frame_path)

print("\nSaved manifest:")
print(manifest_path)

print("\nNEXT GATE:")
print(
    "Visually verify this frame before installing/running Multi-HMR."
)

In [ ]:
# ============================================================
# CELL 16 — MULTI-HMR / SMPL-X ENVIRONMENT PREFLIGHT
# DO NOT INSTALL ANYTHING YET
# ============================================================

import sys
import os
import platform
import shutil
import subprocess
from pathlib import Path

print("=" * 90)
print("MULTI-HMR / SMPL-X ENVIRONMENT PREFLIGHT")
print("=" * 90)

# ------------------------------------------------------------
# Python
# ------------------------------------------------------------

print("\n[PYTHON]")
print("Version    :", sys.version)
print("Executable :", sys.executable)
print("Platform   :", platform.platform())


# ------------------------------------------------------------
# GPU / CUDA
# ------------------------------------------------------------

print("\n[GPU]")

nvidia_smi = shutil.which("nvidia-smi")

if nvidia_smi:
    result = subprocess.run(
        [
            "nvidia-smi",
            "--query-gpu=name,memory.total,driver_version",
            "--format=csv,noheader"
        ],
        capture_output=True,
        text=True
    )

    print(result.stdout.strip())
else:
    print("nvidia-smi NOT FOUND")


# ------------------------------------------------------------
# PyTorch
# ------------------------------------------------------------

print("\n[PYTORCH]")

try:
    import torch

    print("Torch version      :", torch.__version__)
    print("CUDA available     :", torch.cuda.is_available())
    print("Torch CUDA version :", torch.version.cuda)

    if torch.cuda.is_available():

        print(
            "GPU name           :",
            torch.cuda.get_device_name(0)
        )

        props = torch.cuda.get_device_properties(0)

        print(
            "GPU VRAM           :",
            round(props.total_memory / (1024**3), 2),
            "GB"
        )

except Exception as e:
    print("PyTorch check failed:", repr(e))


# ------------------------------------------------------------
# GIT
# ------------------------------------------------------------

print("\n[GIT]")
print("git:", shutil.which("git"))


# ------------------------------------------------------------
# BASIC BUILD TOOLS
# ------------------------------------------------------------

print("\n[BUILD TOOLS]")

for tool in [
    "gcc",
    "g++",
    "cmake",
    "ffmpeg"
]:
    print(
        f"{tool:10s}:",
        shutil.which(tool)
    )


# ------------------------------------------------------------
# CURRENT DISK
# ------------------------------------------------------------

print("\n[DISK]")

usage = shutil.disk_usage("/kaggle/working")

print(
    "Total:",
    round(usage.total / (1024**3), 2),
    "GB"
)

print(
    "Used :",
    round(usage.used / (1024**3), 2),
    "GB"
)

print(
    "Free :",
    round(usage.free / (1024**3), 2),
    "GB"
)


# ------------------------------------------------------------
# INTERNET QUICK CHECK
# ------------------------------------------------------------

print("\n[NETWORK / GITHUB]")

try:

    result = subprocess.run(
        [
            "git",
            "ls-remote",
            "https://github.com/naver/multi-hmr.git",
            "HEAD"
        ],
        capture_output=True,
        text=True,
        timeout=20
    )

    print("Return code:", result.returncode)

    if result.returncode == 0:
        print("GitHub access: PASS")
        print(result.stdout.strip())
    else:
        print("GitHub access: FAIL")
        print(result.stderr[:500])

except Exception as e:
    print("GitHub check error:", repr(e))


print("\n" + "=" * 90)
print("PREFLIGHT COMPLETE")
print("=" * 90)

In [ ]:
# ============================================================
# CELL 17 — FREEZE OFFICIAL MULTI-HMR REPOSITORY
# + CHECK ISOLATED ENVIRONMENT OPTIONS
# ============================================================

import shutil
import subprocess
from pathlib import Path

REPO_DIR = Path("/kaggle/working/multi-hmr")

print("=" * 90)
print("MULTI-HMR REPOSITORY + ENVIRONMENT CHECK")
print("=" * 90)


# ------------------------------------------------------------
# 1. CHECK AVAILABLE ENVIRONMENT TOOLS
# ------------------------------------------------------------

print("\n[ENVIRONMENT TOOLS]")

tools = [
    "python3.9",
    "python3.10",
    "conda",
    "mamba",
    "micromamba",
    "uv",
]

tool_paths = {}

for tool in tools:
    path = shutil.which(tool)
    tool_paths[tool] = path
    print(f"{tool:12s}: {path}")


# ------------------------------------------------------------
# 2. CLONE OFFICIAL REPOSITORY
# ------------------------------------------------------------

print("\n[REPOSITORY]")

if REPO_DIR.exists():
    print("Repository already exists:")
    print(REPO_DIR)

else:
    print("Cloning official NAVER Multi-HMR repository...")

    result = subprocess.run(
        [
            "git",
            "clone",
            "https://github.com/naver/multi-hmr.git",
            str(REPO_DIR),
        ],
        capture_output=True,
        text=True,
        timeout=120,
    )

    print("Clone return code:", result.returncode)

    if result.stdout.strip():
        print(result.stdout[-1000:])

    if result.stderr.strip():
        print(result.stderr[-1000:])

    assert result.returncode == 0, "Multi-HMR clone failed."


# ------------------------------------------------------------
# 3. RECORD EXACT COMMIT
# ------------------------------------------------------------

commit = subprocess.run(
    [
        "git",
        "-C",
        str(REPO_DIR),
        "rev-parse",
        "HEAD",
    ],
    capture_output=True,
    text=True,
    check=True,
).stdout.strip()

print("\nExact repository commit:")
print(commit)


# ------------------------------------------------------------
# 4. CHECK IMPORTANT FILES
# ------------------------------------------------------------

required_files = [
    "README.md",
    "requirements.txt",
    "conda.yaml",
    "demo.py",
    "model.py",
]

print("\n[IMPORTANT FILES]")

for rel_path in required_files:

    path = REPO_DIR / rel_path

    print(
        f"{rel_path:20s} "
        f"{'✅ FOUND' if path.exists() else '❌ MISSING'}"
    )


# ------------------------------------------------------------
# 5. DISPLAY OFFICIAL REQUIREMENTS
# ------------------------------------------------------------

requirements_path = REPO_DIR / "requirements.txt"

print("\n" + "=" * 90)
print("OFFICIAL requirements.txt")
print("=" * 90)

print(
    requirements_path.read_text(
        encoding="utf-8"
    )
)


# ------------------------------------------------------------
# 6. DISPLAY CONDA ENVIRONMENT
# ------------------------------------------------------------

conda_path = REPO_DIR / "conda.yaml"

print("\n" + "=" * 90)
print("OFFICIAL conda.yaml")
print("=" * 90)

print(
    conda_path.read_text(
        encoding="utf-8"
    )
)


# ------------------------------------------------------------
# 7. CHECK SMPL-X DIRECTORY / ASSET
# ------------------------------------------------------------

smplx_dir = REPO_DIR / "models" / "smplx"

smplx_asset = (
    smplx_dir /
    "SMPLX_NEUTRAL.npz"
)

print("\n" + "=" * 90)
print("SMPL-X ASSET CHECK")
print("=" * 90)

print("Expected directory:")
print(smplx_dir)

print("\nExpected asset:")
print(smplx_asset)

print(
    "\nSMPLX_NEUTRAL.npz present:",
    smplx_asset.exists()
)


# ------------------------------------------------------------
# 8. CURRENT REPO SIZE
# ------------------------------------------------------------

repo_size = sum(
    f.stat().st_size
    for f in REPO_DIR.rglob("*")
    if f.is_file()
)

print(
    "\nRepository size:",
    round(repo_size / (1024 ** 2), 2),
    "MB"
)


# ------------------------------------------------------------
# FINAL
# ------------------------------------------------------------

print("\n" + "=" * 90)
print("CELL 17 COMPLETE")
print("=" * 90)

print("Repository :", REPO_DIR)
print("Commit     :", commit)

print("\nDO NOT INSTALL DEPENDENCIES YET.")

In [ ]:
# ============================================================
# CELL 18 — CREATE ISOLATED MULTI-HMR PYTHON 3.9 ENVIRONMENT
# ROBUST KAGGLE VERSION — USE UV, NOT FAKE "MAMBA"
# ============================================================

import subprocess
import shutil
import sys
from pathlib import Path

ENV_ROOT = Path("/kaggle/working/envs")
MULTIHMR_ENV = ENV_ROOT / "multihmr_py39"

ENV_ROOT.mkdir(
    parents=True,
    exist_ok=True
)

print("=" * 90)
print("MULTI-HMR ISOLATED ENVIRONMENT SETUP")
print("=" * 90)

print("\nEnvironment path:")
print(MULTIHMR_ENV)

uv_bin = shutil.which("uv")

print("\nUV:")
print(uv_bin)

assert uv_bin is not None, (
    "❌ uv is not available in this Kaggle runtime."
)


# ------------------------------------------------------------
# 0. CHECK WHETHER A COMPLETE ENV ALREADY EXISTS
# ------------------------------------------------------------

env_python = MULTIHMR_ENV / "bin" / "python"

existing_env_valid = False

if env_python.exists():

    check = subprocess.run(
        [
            str(env_python),
            "-c",
            (
                "import sys; "
                "print(sys.version); "
                "assert sys.version_info[:2] == (3, 9)"
            )
        ],
        capture_output=True,
        text=True
    )

    if check.returncode == 0:
        existing_env_valid = True


# ------------------------------------------------------------
# 1. REMOVE ONLY BROKEN / PARTIAL ENV
# ------------------------------------------------------------

if MULTIHMR_ENV.exists() and not existing_env_valid:

    print("\n⚠️ Partial or invalid environment detected.")
    print("Removing only:")
    print(MULTIHMR_ENV)

    shutil.rmtree(MULTIHMR_ENV)


# ------------------------------------------------------------
# 2. ENSURE PYTHON 3.9 IS AVAILABLE THROUGH UV
# ------------------------------------------------------------

if not existing_env_valid:

    print("\n" + "=" * 90)
    print("INSTALLING / LOCATING PYTHON 3.9")
    print("=" * 90)

    result = subprocess.run(
        [
            uv_bin,
            "python",
            "install",
            "3.9"
        ]
    )

    assert result.returncode == 0, (
        "❌ uv could not install/find Python 3.9."
    )


# ------------------------------------------------------------
# 3. CREATE ISOLATED ENVIRONMENT
# ------------------------------------------------------------

if existing_env_valid:

    print("\n✅ Valid Python 3.9 environment already exists.")
    print("Skipping recreation.")

else:

    print("\n" + "=" * 90)
    print("CREATING PYTHON 3.9 VIRTUAL ENVIRONMENT")
    print("=" * 90)

    result = subprocess.run(
        [
            uv_bin,
            "venv",
            str(MULTIHMR_ENV),
            "--python",
            "3.9",
            "--seed",
        ]
    )

    assert result.returncode == 0, (
        "❌ Failed to create Python 3.9 Multi-HMR environment."
    )


# ------------------------------------------------------------
# 4. VERIFY ENV PYTHON EXISTS
# ------------------------------------------------------------

env_python = MULTIHMR_ENV / "bin" / "python"

assert env_python.exists(), (
    f"❌ Environment Python not found at:\n{env_python}"
)


# ------------------------------------------------------------
# 5. VERIFY PYTHON VERSION
# ------------------------------------------------------------

print("\n" + "=" * 90)
print("ENVIRONMENT VERIFICATION")
print("=" * 90)

python_check = subprocess.run(
    [
        str(env_python),
        "-c",
        """
import sys

print("Python version :", sys.version)
print("Executable     :", sys.executable)

assert sys.version_info.major == 3
assert sys.version_info.minor == 9

print("PYTHON_39_CHECK=PASS")
"""
    ],
    capture_output=True,
    text=True
)

print(python_check.stdout)

if python_check.stderr.strip():
    print("STDERR:")
    print(python_check.stderr)

assert python_check.returncode == 0, (
    "❌ Environment is not running Python 3.9."
)


# ------------------------------------------------------------
# 6. VERIFY PIP
# ------------------------------------------------------------

pip_check = subprocess.run(
    [
        str(env_python),
        "-m",
        "pip",
        "--version"
    ],
    capture_output=True,
    text=True
)

print("\nPip:")
print(
    pip_check.stdout.strip()
    or pip_check.stderr.strip()
)

assert pip_check.returncode == 0, (
    "❌ pip unavailable inside isolated environment."
)


# ------------------------------------------------------------
# 7. STRICT ISOLATION CHECK
# ------------------------------------------------------------

print("\n" + "=" * 90)
print("ISOLATION CHECK")
print("=" * 90)

print("Notebook Python:")
print(sys.executable)

print("\nMulti-HMR Python:")
print(env_python)

assert Path(sys.executable).resolve() != env_python.resolve(), (
    "❌ Multi-HMR environment is not isolated."
)

print("\n✅ Base Kaggle environment remains separate.")


# ------------------------------------------------------------
# 8. SHOW CURRENT ENV PACKAGES
# ------------------------------------------------------------

print("\n" + "=" * 90)
print("INITIAL ENVIRONMENT PACKAGE STATE")
print("=" * 90)

freeze = subprocess.run(
    [
        str(env_python),
        "-m",
        "pip",
        "list"
    ],
    capture_output=True,
    text=True,
    check=True
)

print(freeze.stdout)


# ------------------------------------------------------------
# 9. DISK AFTER ENV CREATION
# ------------------------------------------------------------

usage = shutil.disk_usage("/kaggle/working")

free_gb = usage.free / (1024 ** 3)

print("=" * 90)
print("DISK CHECK")
print("=" * 90)

print(
    "Disk remaining:",
    round(free_gb, 2),
    "GB"
)

assert free_gb > 8, (
    "❌ Less than 8 GB free. Stop before heavyweight dependency installation."
)


# ------------------------------------------------------------
# 10. FINAL STATUS
# ------------------------------------------------------------

print("\n" + "=" * 90)
print("CELL 18 COMPLETE")
print("=" * 90)

print("\nEnvironment:")
print(MULTIHMR_ENV)

print("\nPython:")
print(env_python)

print("""
Expected:
- Python 3.9.x
- pip available
- isolated from Kaggle notebook Python
- NO Multi-HMR dependencies installed yet
""")

print("✅ READY FOR NEXT DEPENDENCY-PREFLIGHT CELL.")
print("\n⚠️ DO NOT pip install the Multi-HMR requirements manually yet.")

In [ ]:
# ============================================================
# CELL 19 — MULTI-HMR DEPENDENCY + REPOSITORY PREFLIGHT
# READ-ONLY AUDIT — INSTALL NOTHING
# ============================================================

import subprocess
import json
import os
import re
import shutil
from pathlib import Path

REPO = Path("/kaggle/working/multi-hmr")
ENV = Path("/kaggle/working/envs/multihmr_py39")
ENV_PYTHON = ENV / "bin" / "python"

EXPECTED_COMMIT = "651fb411e1cbcc626aaa5f38805ecab9cc891f7a"

print("=" * 100)
print("MULTI-HMR DEPENDENCY + REPOSITORY PREFLIGHT")
print("=" * 100)


# ------------------------------------------------------------
# 1. VERIFY REPOSITORY
# ------------------------------------------------------------

print("\n[1] REPOSITORY CHECK")
print("-" * 100)

assert REPO.exists(), f"❌ Repository missing: {REPO}"
assert (REPO / ".git").exists(), f"❌ Not a Git repository: {REPO}"

print("Repository:", REPO)


# ------------------------------------------------------------
# 2. VERIFY FROZEN COMMIT
# ------------------------------------------------------------

commit = subprocess.run(
    ["git", "rev-parse", "HEAD"],
    cwd=REPO,
    capture_output=True,
    text=True,
    check=True
).stdout.strip()

print("HEAD:", commit)
print("Expected:", EXPECTED_COMMIT)

assert commit == EXPECTED_COMMIT, (
    "\n❌ Repository commit drift detected.\n"
    f"Expected: {EXPECTED_COMMIT}\n"
    f"Actual:   {commit}\n"
    "Do not continue until repository state is understood."
)

print("✅ Frozen commit verified.")


# ------------------------------------------------------------
# 3. VERIFY ISOLATED PYTHON
# ------------------------------------------------------------

print("\n[2] ISOLATED ENVIRONMENT")
print("-" * 100)

assert ENV_PYTHON.exists(), (
    f"❌ Isolated Python missing: {ENV_PYTHON}"
)

env_info = subprocess.run(
    [
        str(ENV_PYTHON),
        "-c",
        (
            "import sys, platform; "
            "print('version=' + sys.version.replace('\\n',' ')); "
            "print('executable=' + sys.executable); "
            "print('platform=' + platform.platform())"
        ),
    ],
    capture_output=True,
    text=True,
    check=True,
)

print(env_info.stdout)

print("✅ Environment accessible.")


# ------------------------------------------------------------
# 4. LOCATE DEPENDENCY / PACKAGING FILES
# ------------------------------------------------------------

print("\n[3] PACKAGING / DEPENDENCY FILE DISCOVERY")
print("-" * 100)

candidate_names = {
    "requirements.txt",
    "requirements-dev.txt",
    "environment.yml",
    "environment.yaml",
    "pyproject.toml",
    "setup.py",
    "setup.cfg",
    "Pipfile",
    "Pipfile.lock",
    "poetry.lock",
    "conda.yaml",
    "conda.yml",
}

dependency_files = []

for path in REPO.rglob("*"):
    if not path.is_file():
        continue

    if ".git" in path.parts:
        continue

    if path.name in candidate_names:
        dependency_files.append(path)

    elif path.name.startswith("requirements") and path.suffix in {
        ".txt",
        ".in",
    }:
        dependency_files.append(path)

dependency_files = sorted(set(dependency_files))

if dependency_files:
    for path in dependency_files:
        print(path.relative_to(REPO))
else:
    print("⚠️ No standard dependency file found.")


# ------------------------------------------------------------
# 5. PRINT DEPENDENCY FILE CONTENTS
# ------------------------------------------------------------

print("\n[4] DEPENDENCY FILE CONTENTS")
print("-" * 100)

MAX_CHARS_PER_FILE = 30000

for path in dependency_files:

    print("\n" + "=" * 100)
    print(f"FILE: {path.relative_to(REPO)}")
    print("=" * 100)

    try:
        text = path.read_text(
            encoding="utf-8",
            errors="replace"
        )
    except Exception as exc:
        print("READ ERROR:", repr(exc))
        continue

    if len(text) > MAX_CHARS_PER_FILE:
        print(text[:MAX_CHARS_PER_FILE])
        print(
            f"\n... TRUNCATED "
            f"({len(text) - MAX_CHARS_PER_FILE} chars omitted)"
        )
    else:
        print(text)


# ------------------------------------------------------------
# 6. SEARCH REPOSITORY FOR HIGH-RISK DEPENDENCIES
# ------------------------------------------------------------

print("\n[5] HIGH-RISK DEPENDENCY REFERENCES")
print("-" * 100)

patterns = [
    r"torch(?:vision|audio)?[<>=~! ]",
    r"pytorch",
    r"cuda",
    r"smplx",
    r"pytorch3d",
    r"xformers",
    r"detectron",
    r"mmcv",
    r"mmpose",
    r"timm",
    r"einops",
    r"triton",
    r"chumpy",
    r"numpy",
    r"scipy",
    r"opencv",
]

compiled = [
    re.compile(pattern, re.IGNORECASE)
    for pattern in patterns
]

allowed_suffixes = {
    ".txt",
    ".md",
    ".yaml",
    ".yml",
    ".toml",
    ".cfg",
    ".ini",
    ".py",
    ".sh",
}

matches = []

for path in REPO.rglob("*"):

    if not path.is_file():
        continue

    if ".git" in path.parts:
        continue

    if path.suffix.lower() not in allowed_suffixes:
        continue

    # skip potentially huge generated/vendor files
    try:
        if path.stat().st_size > 2_000_000:
            continue
    except OSError:
        continue

    try:
        lines = path.read_text(
            encoding="utf-8",
            errors="ignore",
        ).splitlines()
    except Exception:
        continue

    for lineno, line in enumerate(lines, 1):

        if any(regex.search(line) for regex in compiled):

            matches.append(
                (
                    str(path.relative_to(REPO)),
                    lineno,
                    line.strip(),
                )
            )


MAX_MATCHES = 300

for file_name, lineno, line in matches[:MAX_MATCHES]:
    print(f"{file_name}:{lineno}: {line}")

if len(matches) > MAX_MATCHES:
    print(
        f"\n... {len(matches) - MAX_MATCHES} "
        "additional dependency-related matches omitted."
    )

print(
    "\nTotal dependency-related references:",
    len(matches)
)


# ------------------------------------------------------------
# 7. README INSTALLATION INSTRUCTIONS
# ------------------------------------------------------------

print("\n[6] README INSTALLATION / ENVIRONMENT SECTIONS")
print("-" * 100)

readmes = sorted(
    [
        path
        for path in REPO.glob("README*")
        if path.is_file()
    ]
)

if not readmes:

    print("⚠️ No root README found.")

else:

    keywords = (
        "install",
        "environment",
        "requirement",
        "python",
        "torch",
        "cuda",
        "smpl",
        "setup",
        "checkpoint",
        "model",
    )

    for readme in readmes:

        print("\n" + "=" * 100)
        print("README:", readme.name)
        print("=" * 100)

        lines = readme.read_text(
            encoding="utf-8",
            errors="replace",
        ).splitlines()

        sections = []

        for i, line in enumerate(lines):

            lower = line.lower()

            if (
                line.lstrip().startswith("#")
                and any(keyword in lower for keyword in keywords)
            ):
                start = i
                end = min(i + 60, len(lines))

                sections.append(
                    "\n".join(lines[start:end])
                )

        if sections:
            for section in sections[:10]:
                print(section)
                print("\n" + "-" * 100)
        else:
            print(
                "No obvious installation/environment "
                "heading detected."
            )


# ------------------------------------------------------------
# 8. LOCATE CHECKPOINT / MODEL REFERENCES
# ------------------------------------------------------------

print("\n[7] CHECKPOINT / MODEL FILE INVENTORY")
print("-" * 100)

model_suffixes = {
    ".pth",
    ".pt",
    ".ckpt",
    ".bin",
    ".safetensors",
    ".npz",
    ".pkl",
}

model_files = []

for path in REPO.rglob("*"):

    if (
        path.is_file()
        and path.suffix.lower() in model_suffixes
        and ".git" not in path.parts
    ):
        model_files.append(path)

if model_files:

    for path in sorted(model_files):

        size_mb = path.stat().st_size / (1024 ** 2)

        print(
            f"{path.relative_to(REPO)} "
            f"({size_mb:.2f} MB)"
        )

else:
    print("No model/checkpoint files currently present.")


# ------------------------------------------------------------
# 9. SMPL-X ASSET GATE
# ------------------------------------------------------------

print("\n[8] SMPL-X ASSET GATE")
print("-" * 100)

expected_smplx = (
    REPO
    / "models"
    / "smplx"
    / "SMPLX_NEUTRAL.npz"
)

print("Expected path:")
print(expected_smplx)

if expected_smplx.exists():

    print(
        f"✅ Present "
        f"({expected_smplx.stat().st_size / (1024**2):.2f} MB)"
    )

else:

    print("⚠️ MISSING")
    print(
        "This is NOT fatal for dependency-preflight."
    )
    print(
        "Do not create a fake placeholder asset."
    )


# ------------------------------------------------------------
# 10. GPU / DRIVER INFORMATION FROM BASE RUNTIME
# ------------------------------------------------------------

print("\n[9] KAGGLE GPU / DRIVER")
print("-" * 100)

nvidia_smi = shutil.which("nvidia-smi")

if nvidia_smi:

    gpu = subprocess.run(
        [
            nvidia_smi,
            "--query-gpu="
            "name,"
            "driver_version,"
            "memory.total,"
            "memory.free,"
            "compute_cap",
            "--format=csv,noheader",
        ],
        capture_output=True,
        text=True,
    )

    if gpu.returncode == 0:
        print(gpu.stdout.strip())
    else:
        print("nvidia-smi query failed:")
        print(gpu.stderr)

else:
    print("⚠️ nvidia-smi unavailable.")


# ------------------------------------------------------------
# 11. DISK CHECK
# ------------------------------------------------------------

print("\n[10] STORAGE")
print("-" * 100)

usage = shutil.disk_usage("/kaggle/working")

print(
    "Free:",
    round(usage.free / (1024 ** 3), 2),
    "GB"
)

print(
    "Used:",
    round(usage.used / (1024 ** 3), 2),
    "GB"
)

print(
    "Total:",
    round(usage.total / (1024 ** 3), 2),
    "GB"
)


# ------------------------------------------------------------
# 12. CURRENT ENV PACKAGE STATE
# ------------------------------------------------------------

print("\n[11] CURRENT ISOLATED ENV PACKAGE STATE")
print("-" * 100)

packages = subprocess.run(
    [
        str(ENV_PYTHON),
        "-m",
        "pip",
        "list",
    ],
    capture_output=True,
    text=True,
    check=True,
)

print(packages.stdout)


# ------------------------------------------------------------
# 13. FINAL STATUS
# ------------------------------------------------------------

print("=" * 100)
print("CELL 19 COMPLETE — READ-ONLY PREFLIGHT")
print("=" * 100)

print(
    """
No Multi-HMR dependency installation was performed.

Next decision will be based on:
1. repository-declared Python version
2. exact Torch / TorchVision requirements
3. CUDA compatibility
4. compiled dependencies such as PyTorch3D
5. SMPL-X package requirements
6. checkpoint expectations
7. available Kaggle T4 driver/runtime
"""
)

print("✅ SAFE TO REVIEW OUTPUT BEFORE INSTALLATION.")

In [ ]:
# ============================================================
# CELL 20 — INSTALL + VERIFY PYTORCH 2.0.1 CUDA 11.8 FOUNDATION
# ISOLATED MULTI-HMR ENV ONLY
# ============================================================

import subprocess
import shutil
from pathlib import Path

ENV = Path("/kaggle/working/envs/multihmr_py39")
PYTHON = ENV / "bin" / "python"

print("=" * 100)
print("MULTI-HMR — PYTORCH GPU FOUNDATION")
print("=" * 100)

assert PYTHON.exists(), f"Missing isolated Python: {PYTHON}"

print("\nEnvironment:")
print(ENV)

print("\nPython:")
print(PYTHON)


# ------------------------------------------------------------
# 1. DISK PRECHECK
# ------------------------------------------------------------

usage = shutil.disk_usage("/kaggle/working")
free_gb = usage.free / (1024 ** 3)

print("\nDisk free before install:")
print(round(free_gb, 2), "GB")

assert free_gb > 10, (
    "❌ Less than 10 GB free. Stop before installing PyTorch."
)


# ------------------------------------------------------------
# 2. INSTALL MATCHED PYTORCH 2.0.1 STACK
# ------------------------------------------------------------

print("\n" + "=" * 100)
print("INSTALLING PYTORCH 2.0.1 + CUDA 11.8 WHEELS")
print("=" * 100)

cmd = [
    str(PYTHON),
    "-m",
    "pip",
    "install",

    "torch==2.0.1",
    "torchvision==0.15.2",
    "torchaudio==2.0.2",

    "--index-url",
    "https://download.pytorch.org/whl/cu118",
]

print("\nCommand:")
print(" ".join(cmd))

result = subprocess.run(cmd)

assert result.returncode == 0, (
    "❌ PyTorch installation failed."
)


# ------------------------------------------------------------
# 3. VERIFY IMPORT + CUDA
# ------------------------------------------------------------

print("\n" + "=" * 100)
print("PYTORCH CUDA VERIFICATION")
print("=" * 100)

verify_code = r'''
import sys
import torch
import torchvision

print("Python:")
print(sys.version)

print("\nTorch:")
print(torch.__version__)

print("\nTorchVision:")
print(torchvision.__version__)

print("\nTorch CUDA build:")
print(torch.version.cuda)

print("\nCUDA available:")
print(torch.cuda.is_available())

print("\nCUDA device count:")
print(torch.cuda.device_count())

if torch.cuda.is_available():
    print("\nGPU:")
    print(torch.cuda.get_device_name(0))

    props = torch.cuda.get_device_properties(0)

    print("\nCompute capability:")
    print(f"{props.major}.{props.minor}")

    print("\nVRAM total GB:")
    print(round(props.total_memory / (1024 ** 3), 2))

assert torch.__version__.startswith("2.0.1")
assert torchvision.__version__.startswith("0.15.2")
assert torch.version.cuda == "11.8"
assert torch.cuda.is_available()

print("\nTORCH_GPU_CHECK=PASS")
'''

verify = subprocess.run(
    [
        str(PYTHON),
        "-c",
        verify_code,
    ],
    capture_output=True,
    text=True
)

print(verify.stdout)

if verify.stderr.strip():
    print("\nSTDERR:")
    print(verify.stderr)

assert verify.returncode == 0, (
    "❌ Torch installed, but GPU verification failed."
)


# ------------------------------------------------------------
# 4. BASIC GPU COMPUTE TEST
# ------------------------------------------------------------

print("\n" + "=" * 100)
print("GPU COMPUTE SMOKE TEST")
print("=" * 100)

gpu_test = r'''
import torch

device = torch.device("cuda")

a = torch.randn(
    1024,
    1024,
    device=device
)

b = torch.randn(
    1024,
    1024,
    device=device
)

torch.cuda.synchronize()

c = a @ b

torch.cuda.synchronize()

print("shape:", tuple(c.shape))
print("device:", c.device)
print("finite:", bool(torch.isfinite(c).all()))

assert c.is_cuda
assert torch.isfinite(c).all()

print("GPU_MATMUL_CHECK=PASS")
'''

test = subprocess.run(
    [
        str(PYTHON),
        "-c",
        gpu_test,
    ],
    capture_output=True,
    text=True
)

print(test.stdout)

if test.stderr.strip():
    print("\nSTDERR:")
    print(test.stderr)

assert test.returncode == 0, (
    "❌ CUDA matrix multiplication smoke test failed."
)


# ------------------------------------------------------------
# 5. PACKAGE SNAPSHOT
# ------------------------------------------------------------

print("\n" + "=" * 100)
print("INSTALLED FOUNDATION PACKAGES")
print("=" * 100)

snapshot = subprocess.run(
    [
        str(PYTHON),
        "-m",
        "pip",
        "list",
    ],
    capture_output=True,
    text=True,
    check=True
)

print(snapshot.stdout)


# ------------------------------------------------------------
# 6. DISK AFTER INSTALL
# ------------------------------------------------------------

usage = shutil.disk_usage("/kaggle/working")
free_after = usage.free / (1024 ** 3)

print("=" * 100)
print("DISK AFTER INSTALL")
print("=" * 100)

print(
    round(free_after, 2),
    "GB"
)


# ------------------------------------------------------------
# FINAL
# ------------------------------------------------------------

print("\n" + "=" * 100)
print("CELL 20 COMPLETE")
print("=" * 100)

print("""
Required PASS conditions:

torch        = 2.0.1
torchvision  = 0.15.2
CUDA build   = 11.8
CUDA usable  = True
GPU          = Tesla T4
GPU matmul   = PASS

Do NOT install xformers or the remaining requirements yet.
""")

print("✅ PYTORCH FOUNDATION READY.")

In [ ]:
# ============================================================
# CELL 21 — PIN NUMPY 1.22.4 + VERIFY TORCH/NUMPY INTEROP
# ============================================================

import subprocess
from pathlib import Path

ENV = Path("/kaggle/working/envs/multihmr_py39")
PYTHON = ENV / "bin" / "python"

print("=" * 100)
print("MULTI-HMR — NUMPY COMPATIBILITY FIX")
print("=" * 100)

assert PYTHON.exists()


# ------------------------------------------------------------
# 1. PIN REPOSITORY-EXPECTED NUMPY
# ------------------------------------------------------------

print("\nInstalling numpy==1.22.4 ...")

result = subprocess.run(
    [
        str(PYTHON),
        "-m",
        "pip",
        "install",
        "--force-reinstall",
        "numpy==1.22.4",
    ]
)

assert result.returncode == 0, (
    "❌ NumPy downgrade failed."
)


# ------------------------------------------------------------
# 2. VERIFY VERSION
# ------------------------------------------------------------

print("\n" + "=" * 100)
print("VERSION CHECK")
print("=" * 100)

verify = subprocess.run(
    [
        str(PYTHON),
        "-c",
        r'''
import numpy as np
import torch
import torchvision

print("NumPy:", np.__version__)
print("Torch:", torch.__version__)
print("TorchVision:", torchvision.__version__)
print("CUDA:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())

assert np.__version__ == "1.22.4"
assert torch.__version__.startswith("2.0.1")
assert torchvision.__version__.startswith("0.15.2")
assert torch.cuda.is_available()

print("VERSION_CHECK=PASS")
'''
    ],
    capture_output=True,
    text=True
)

print(verify.stdout)

if verify.stderr.strip():
    print("STDERR:")
    print(verify.stderr)

assert verify.returncode == 0, (
    "❌ Version compatibility verification failed."
)


# ------------------------------------------------------------
# 3. TORCH <-> NUMPY ROUNDTRIP
# ------------------------------------------------------------

print("\n" + "=" * 100)
print("TORCH ↔ NUMPY INTEROP TEST")
print("=" * 100)

interop = subprocess.run(
    [
        str(PYTHON),
        "-c",
        r'''
import numpy as np
import torch

a = np.random.randn(4, 5).astype(np.float32)

t = torch.from_numpy(a)

b = t.numpy()

print("numpy shape:", a.shape)
print("tensor shape:", tuple(t.shape))
print("roundtrip shape:", b.shape)

print("same:", np.allclose(a, b))

assert np.allclose(a, b)

print("TORCH_NUMPY_INTEROP=PASS")
'''
    ],
    capture_output=True,
    text=True
)

print(interop.stdout)

if interop.stderr.strip():
    print("STDERR:")
    print(interop.stderr)

assert interop.returncode == 0, (
    "❌ Torch/NumPy interoperability failed."
)


# ------------------------------------------------------------
# 4. CUDA + NUMPY BRIDGE TEST
# ------------------------------------------------------------

print("\n" + "=" * 100)
print("CUDA → CPU → NUMPY TEST")
print("=" * 100)

cuda_test = subprocess.run(
    [
        str(PYTHON),
        "-c",
        r'''
import numpy as np
import torch

x = torch.randn(
    128,
    128,
    device="cuda"
)

y = x @ x.T

arr = y.detach().cpu().numpy()

print("GPU:", torch.cuda.get_device_name(0))
print("array:", arr.shape, arr.dtype)
print("finite:", np.isfinite(arr).all())

assert arr.shape == (128, 128)
assert np.isfinite(arr).all()

print("CUDA_NUMPY_BRIDGE=PASS")
'''
    ],
    capture_output=True,
    text=True
)

print(cuda_test.stdout)

if cuda_test.stderr.strip():
    print("STDERR:")
    print(cuda_test.stderr)

assert cuda_test.returncode == 0, (
    "❌ CUDA → NumPy bridge failed."
)


# ------------------------------------------------------------
# 5. FINAL PACKAGE SNAPSHOT
# ------------------------------------------------------------

print("\n" + "=" * 100)
print("FOUNDATION PACKAGE SNAPSHOT")
print("=" * 100)

snapshot = subprocess.run(
    [
        str(PYTHON),
        "-m",
        "pip",
        "list",
    ],
    capture_output=True,
    text=True,
    check=True
)

print(snapshot.stdout)


print("\n" + "=" * 100)
print("CELL 21 COMPLETE")
print("=" * 100)

print("""
Expected:

numpy                  1.22.4
torch                  2.0.1+cu118
torchvision            0.15.2+cu118

VERSION_CHECK=PASS
TORCH_NUMPY_INTEROP=PASS
CUDA_NUMPY_BRIDGE=PASS
""")

print("✅ TORCH FOUNDATION NOW NUMPY-COMPATIBLE.")

In [ ]:
# ============================================================
# CELL 22 — INSTALL + VERIFY XFORMERS 0.0.20
# KEEP TORCH 2.0.1 FROZEN
# ============================================================

import subprocess
from pathlib import Path

ENV = Path("/kaggle/working/envs/multihmr_py39")
PYTHON = ENV / "bin" / "python"

print("=" * 100)
print("MULTI-HMR — XFORMERS INSTALLATION")
print("=" * 100)

assert PYTHON.exists()


# ------------------------------------------------------------
# 1. RECORD TORCH STATE BEFORE INSTALL
# ------------------------------------------------------------

print("\n[1] TORCH STATE BEFORE XFORMERS")
print("-" * 100)

before = subprocess.run(
    [
        str(PYTHON),
        "-c",
        r'''
import torch
import numpy as np

print("torch =", torch.__version__)
print("cuda  =", torch.version.cuda)
print("numpy =", np.__version__)

assert torch.__version__.startswith("2.0.1")
assert np.__version__ == "1.22.4"

print("PREINSTALL_STATE=PASS")
'''
    ],
    capture_output=True,
    text=True
)

print(before.stdout)

if before.stderr.strip():
    print("STDERR:")
    print(before.stderr)

assert before.returncode == 0


# ------------------------------------------------------------
# 2. INSTALL EXACT XFORMERS VERSION WITHOUT DEP RESOLUTION
# ------------------------------------------------------------

print("\n[2] INSTALLING xformers==0.0.20")
print("-" * 100)

cmd = [
    str(PYTHON),
    "-m",
    "pip",
    "install",
    "--no-deps",
    "xformers==0.0.20",
]

print("Command:")
print(" ".join(cmd))

result = subprocess.run(cmd)

assert result.returncode == 0, (
    "❌ xformers installation failed."
)


# ------------------------------------------------------------
# 3. VERIFY CORE VERSIONS DID NOT DRIFT
# ------------------------------------------------------------

print("\n[3] VERSION DRIFT CHECK")
print("-" * 100)

check = subprocess.run(
    [
        str(PYTHON),
        "-c",
        r'''
import torch
import numpy as np
import xformers

print("torch     =", torch.__version__)
print("cuda      =", torch.version.cuda)
print("numpy     =", np.__version__)
print("xformers  =", xformers.__version__)

assert torch.__version__.startswith("2.0.1")
assert torch.version.cuda == "11.8"
assert np.__version__ == "1.22.4"
assert xformers.__version__ == "0.0.20"

print("VERSION_DRIFT_CHECK=PASS")
'''
    ],
    capture_output=True,
    text=True
)

print(check.stdout)

if check.stderr.strip():
    print("\nSTDERR:")
    print(check.stderr)

assert check.returncode == 0, (
    "❌ Version drift or xformers import failure."
)


# ------------------------------------------------------------
# 4. INSPECT XFORMERS BUILD INFO
# ------------------------------------------------------------

print("\n[4] XFORMERS BUILD INFORMATION")
print("-" * 100)

info = subprocess.run(
    [
        str(PYTHON),
        "-m",
        "xformers.info",
    ],
    capture_output=True,
    text=True
)

print(info.stdout)

if info.stderr.strip():
    print("\nSTDERR:")
    print(info.stderr)

assert info.returncode == 0, (
    "❌ `python -m xformers.info` failed."
)


# ------------------------------------------------------------
# 5. CUDA MEMORY-EFFICIENT ATTENTION SMOKE TEST
# ------------------------------------------------------------

print("\n[5] MEMORY-EFFICIENT ATTENTION CUDA TEST")
print("-" * 100)

attention_test = subprocess.run(
    [
        str(PYTHON),
        "-c",
        r'''
import torch
from xformers.ops import memory_efficient_attention

assert torch.cuda.is_available()

device = "cuda"

# Shape:
# batch, sequence, heads, head_dim
q = torch.randn(
    2,
    128,
    8,
    64,
    device=device,
    dtype=torch.float16
)

k = torch.randn(
    2,
    128,
    8,
    64,
    device=device,
    dtype=torch.float16
)

v = torch.randn(
    2,
    128,
    8,
    64,
    device=device,
    dtype=torch.float16
)

torch.cuda.synchronize()

out = memory_efficient_attention(q, k, v)

torch.cuda.synchronize()

print("output shape:", tuple(out.shape))
print("dtype:", out.dtype)
print("device:", out.device)
print("finite:", bool(torch.isfinite(out).all()))

assert out.is_cuda
assert tuple(out.shape) == tuple(q.shape)
assert torch.isfinite(out).all()

print("XFORMERS_ATTENTION_CHECK=PASS")
'''
    ],
    capture_output=True,
    text=True
)

print(attention_test.stdout)

if attention_test.stderr.strip():
    print("\nSTDERR:")
    print(attention_test.stderr)

assert attention_test.returncode == 0, (
    "❌ xFormers CUDA attention smoke test failed."
)


# ------------------------------------------------------------
# 6. PACKAGE FREEZE FOR IMPORTANT COMPONENTS
# ------------------------------------------------------------

print("\n[6] FOUNDATION PACKAGE CHECK")
print("-" * 100)

snapshot = subprocess.run(
    [
        str(PYTHON),
        "-m",
        "pip",
        "list",
    ],
    capture_output=True,
    text=True,
    check=True
)

important = {
    "torch",
    "torchvision",
    "torchaudio",
    "numpy",
    "xformers",
    "triton",
}

for line in snapshot.stdout.splitlines():

    lower = line.lower()

    if any(
        lower.startswith(pkg + " ")
        for pkg in important
    ):
        print(line)


# ------------------------------------------------------------
# FINAL STATUS
# ------------------------------------------------------------

print("\n" + "=" * 100)
print("CELL 22 COMPLETE")
print("=" * 100)

print("""
Required:

torch       2.0.1+cu118
numpy       1.22.4
xformers    0.0.20

VERSION_DRIFT_CHECK=PASS
XFORMERS_ATTENTION_CHECK=PASS
""")

print("✅ XFORMERS FOUNDATION READY.")

In [ ]:
# ============================================================
# CELL 23 — INSTALL MULTI-HMR INFERENCE DEPENDENCIES
# TRAINING / GRADIO EXTRAS EXCLUDED
# ============================================================

import subprocess
import shutil
from pathlib import Path

ENV = Path("/kaggle/working/envs/multihmr_py39")
PYTHON = ENV / "bin" / "python"

print("=" * 100)
print("MULTI-HMR — CORE INFERENCE DEPENDENCY INSTALLATION")
print("=" * 100)

assert PYTHON.exists()


# ------------------------------------------------------------
# 1. PRE-INSTALL FOUNDATION CHECK
# ------------------------------------------------------------

print("\n[1] FOUNDATION CHECK")
print("-" * 100)

precheck = subprocess.run(
    [
        str(PYTHON),
        "-c",
        r'''
import torch
import torchvision
import numpy
import xformers

print("torch      =", torch.__version__)
print("torchvision=", torchvision.__version__)
print("numpy      =", numpy.__version__)
print("xformers   =", xformers.__version__)

assert torch.__version__.startswith("2.0.1")
assert torchvision.__version__.startswith("0.15.2")
assert numpy.__version__ == "1.22.4"
assert xformers.__version__ == "0.0.20"

print("FOUNDATION_PRECHECK=PASS")
'''
    ],
    capture_output=True,
    text=True
)

print(precheck.stdout)

if precheck.stderr.strip():
    print("STDERR:")
    print(precheck.stderr)

assert precheck.returncode == 0


# ------------------------------------------------------------
# 2. DISK CHECK
# ------------------------------------------------------------

usage = shutil.disk_usage("/kaggle/working")
free_gb = usage.free / (1024 ** 3)

print("\nDisk free:")
print(round(free_gb, 2), "GB")

assert free_gb > 8, (
    "❌ Less than 8 GB free."
)


# ------------------------------------------------------------
# 3. CREATE CONSTRAINT FILE
# ------------------------------------------------------------

print("\n[2] WRITING FOUNDATION CONSTRAINTS")
print("-" * 100)

constraints = ENV / "multihmr_foundation_constraints.txt"

constraints.write_text(
    """
torch==2.0.1
torchvision==0.15.2
torchaudio==2.0.2
numpy==1.22.4
xformers==0.0.20
""".strip()
)

print(constraints.read_text())


# ------------------------------------------------------------
# 4. INSTALL REPOSITORY INFERENCE DEPENDENCIES
# ------------------------------------------------------------

print("\n[3] INSTALLING CORE INFERENCE PACKAGES")
print("-" * 100)

packages = [
    "roma",
    "pillow==10.0.1",
    "pyrender==0.1.45",
    "einops==0.6.1",
    "smplx",
    "pyvista==0.42.3",
    "trimesh==3.22.3",
    "pyglet==1.5.24",
    "tqdm==4.65.0",
]

cmd = [
    str(PYTHON),
    "-m",
    "pip",
    "install",
    "-c",
    str(constraints),
    *packages,
]

print("Command:")
print(" ".join(cmd))

result = subprocess.run(cmd)

assert result.returncode == 0, (
    "❌ Core inference dependency installation failed."
)


# ------------------------------------------------------------
# 5. VERIFY FOUNDATION DID NOT DRIFT
# ------------------------------------------------------------

print("\n[4] FOUNDATION DRIFT CHECK")
print("-" * 100)

drift = subprocess.run(
    [
        str(PYTHON),
        "-c",
        r'''
import torch
import torchvision
import numpy
import xformers

print("torch       =", torch.__version__)
print("torchvision =", torchvision.__version__)
print("numpy       =", numpy.__version__)
print("xformers    =", xformers.__version__)

assert torch.__version__.startswith("2.0.1")
assert torchvision.__version__.startswith("0.15.2")
assert numpy.__version__ == "1.22.4"
assert xformers.__version__ == "0.0.20"

print("FOUNDATION_DRIFT_CHECK=PASS")
'''
    ],
    capture_output=True,
    text=True
)

print(drift.stdout)

if drift.stderr.strip():
    print("STDERR:")
    print(drift.stderr)

assert drift.returncode == 0, (
    "❌ Foundation package drift detected."
)


# ------------------------------------------------------------
# 6. IMPORT CORE INFERENCE DEPENDENCIES
# ------------------------------------------------------------

print("\n[5] CORE PACKAGE IMPORT TEST")
print("-" * 100)

import_test = subprocess.run(
    [
        str(PYTHON),
        "-c",
        r'''
import roma
import PIL
import pyrender
import einops
import smplx
import pyvista
import trimesh
import pyglet
import tqdm

print("roma       = OK")
print("PIL        =", PIL.__version__)
print("pyrender   =", getattr(pyrender, "__version__", "imported"))
print("einops     =", einops.__version__)
print("smplx      =", getattr(smplx, "__version__", "imported"))
print("pyvista    =", pyvista.__version__)
print("trimesh    =", trimesh.__version__)
print("pyglet     =", pyglet.version)
print("tqdm       =", tqdm.__version__)

print("CORE_IMPORT_CHECK=PASS")
'''
    ],
    capture_output=True,
    text=True
)

print(import_test.stdout)

if import_test.stderr.strip():
    print("\nSTDERR:")
    print(import_test.stderr)

assert import_test.returncode == 0, (
    "❌ One or more core inference imports failed."
)


# ------------------------------------------------------------
# 7. SMPL-X PACKAGE FUNCTIONAL SMOKE TEST
#    DOES NOT REQUIRE LICENSED MODEL FILE YET
# ------------------------------------------------------------

print("\n[6] SMPL-X PYTHON PACKAGE TEST")
print("-" * 100)

smplx_test = subprocess.run(
    [
        str(PYTHON),
        "-c",
        r'''
import smplx
from smplx.joint_names import JOINT_NAMES

print("SMPL-X import: PASS")
print("Joint-name count:", len(JOINT_NAMES))
print("First joints:", JOINT_NAMES[:10])

assert len(JOINT_NAMES) > 0

print("SMPLX_PACKAGE_CHECK=PASS")
'''
    ],
    capture_output=True,
    text=True
)

print(smplx_test.stdout)

if smplx_test.stderr.strip():
    print("\nSTDERR:")
    print(smplx_test.stderr)

assert smplx_test.returncode == 0, (
    "❌ SMPL-X Python package smoke test failed."
)


# ------------------------------------------------------------
# 8. XFORMERS RECHECK AFTER DEP INSTALL
# ------------------------------------------------------------

print("\n[7] XFORMERS CUDA REGRESSION CHECK")
print("-" * 100)

xformers_test = subprocess.run(
    [
        str(PYTHON),
        "-c",
        r'''
import torch
from xformers.ops import memory_efficient_attention

q = torch.randn(
    1, 64, 4, 64,
    dtype=torch.float16,
    device="cuda"
)

k = torch.randn_like(q)
v = torch.randn_like(q)

out = memory_efficient_attention(q, k, v)

torch.cuda.synchronize()

print("shape:", tuple(out.shape))
print("finite:", bool(torch.isfinite(out).all()))

assert torch.isfinite(out).all()

print("XFORMERS_REGRESSION_CHECK=PASS")
'''
    ],
    capture_output=True,
    text=True
)

print(xformers_test.stdout)

if xformers_test.stderr.strip():
    print("\nSTDERR:")
    print(xformers_test.stderr)

assert xformers_test.returncode == 0


# ------------------------------------------------------------
# 9. PIP DEPENDENCY CONSISTENCY
# ------------------------------------------------------------

print("\n[8] PIP DEPENDENCY CONSISTENCY")
print("-" * 100)

pip_check = subprocess.run(
    [
        str(PYTHON),
        "-m",
        "pip",
        "check"
    ],
    capture_output=True,
    text=True
)

print(
    pip_check.stdout.strip()
    or "pip check produced no stdout."
)

if pip_check.stderr.strip():
    print("STDERR:")
    print(pip_check.stderr)

if pip_check.returncode == 0:
    print("PIP_CHECK=PASS")
else:
    print("⚠️ PIP_CHECK=NONZERO")
    print(
        "Do not automatically repair anything yet. "
        "We need to inspect the exact conflict first."
    )


# ------------------------------------------------------------
# 10. FINAL PACKAGE SNAPSHOT
# ------------------------------------------------------------

print("\n[9] IMPORTANT PACKAGE SNAPSHOT")
print("-" * 100)

snapshot = subprocess.run(
    [
        str(PYTHON),
        "-m",
        "pip",
        "list",
    ],
    capture_output=True,
    text=True,
    check=True
)

important = (
    "torch",
    "torchvision",
    "torchaudio",
    "numpy",
    "xformers",
    "roma",
    "pillow",
    "pyrender",
    "einops",
    "smplx",
    "pyvista",
    "trimesh",
    "pyglet",
    "tqdm",
    "scipy",
    "vtk",
)

for line in snapshot.stdout.splitlines():

    lower = line.lower()

    if any(
        lower.startswith(name + " ")
        for name in important
    ):
        print(line)


# ------------------------------------------------------------
# 11. DISK
# ------------------------------------------------------------

usage = shutil.disk_usage("/kaggle/working")

print("\nDisk remaining:")
print(
    round(
        usage.free / (1024 ** 3),
        2
    ),
    "GB"
)


# ------------------------------------------------------------
# FINAL
# ------------------------------------------------------------

print("\n" + "=" * 100)
print("CELL 23 COMPLETE")
print("=" * 100)

print("""
Required PASS:

FOUNDATION_PRECHECK=PASS
FOUNDATION_DRIFT_CHECK=PASS
CORE_IMPORT_CHECK=PASS
SMPLX_PACKAGE_CHECK=PASS
XFORMERS_REGRESSION_CHECK=PASS

PIP_CHECK should preferably PASS.

Still intentionally NOT installed:
- gradio
- spaces
- tensorboard
- chumpy
- anny

Still NOT required yet:
- SMPLX_NEUTRAL.npz
- Multi-HMR checkpoint
""")

print("✅ CORE MULTI-HMR INFERENCE DEPENDENCIES READY.")

In [ ]:
# ============================================================
# NEXT CELL — LOCATE + INSTALL SMPLX_NEUTRAL.npz
# ============================================================

from pathlib import Path
import shutil

TARGET = Path(
    "/kaggle/working/multi-hmr/models/smplx/SMPLX_NEUTRAL.npz"
)

print("=" * 90)
print("SEARCHING KAGGLE INPUTS FOR SMPLX_NEUTRAL.npz")
print("=" * 90)

matches = list(
    Path("/kaggle/input").rglob("SMPLX_NEUTRAL.npz")
)

print("\nMatches found:", len(matches))

for i, path in enumerate(matches):
    print(f"[{i}] {path}")
    print(
        "    Size:",
        round(path.stat().st_size / (1024**2), 2),
        "MB"
    )

assert len(matches) > 0, (
    "❌ SMPLX_NEUTRAL.npz not found under /kaggle/input.\n"
    "Attach your private Kaggle Model to this notebook first."
)

# If exactly one exists, use it automatically.
# If multiple exist, inspect printed paths before continuing.
assert len(matches) == 1, (
    "⚠️ Multiple SMPLX_NEUTRAL.npz files found. "
    "Choose the correct one manually."
)

SOURCE = matches[0]

TARGET.parent.mkdir(
    parents=True,
    exist_ok=True
)

shutil.copy2(
    SOURCE,
    TARGET
)

assert TARGET.exists()

print("\n✅ SMPL-X asset copied.")

print("\nSource:")
print(SOURCE)

print("\nTarget:")
print(TARGET)

print(
    "\nTarget size:",
    round(TARGET.stat().st_size / (1024**2), 2),
    "MB"
)

print("\nSMPLX_ASSET_READY=PASS")

In [ ]:
# ============================================================
# INSPECT BOTH SMPLX_NEUTRAL.npz FILES
# CHOOSE THE CORRECT ONE FOR MULTI-HMR
# ============================================================

from pathlib import Path
import numpy as np

FILES = [
    Path(
        "/kaggle/input/models/dravvalt/"
        "smplx-neutral-v1-1-private/pytorch/default/1/"
        "models/smplx/SMPLX_NEUTRAL.npz"
    ),
    Path(
        "/kaggle/input/models/dravvalt/"
        "smplx-neutral-v1-1-private/pytorch/default/1/"
        "models/smplx/smplx_npz/SMPLX_NEUTRAL.npz"
    ),
]

print("=" * 100)
print("SMPL-X NPZ COMPARISON")
print("=" * 100)

for i, path in enumerate(FILES):

    print(f"\nFILE [{i}]")
    print("-" * 100)
    print("Path:", path)
    print("Size MB:", round(path.stat().st_size / (1024**2), 2))

    data = np.load(path, allow_pickle=True)

    print("\nKeys:")
    print(sorted(data.files))

    print("\nImportant shapes:")

    for key in [
        "v_template",
        "shapedirs",
        "expr_dirs",
        "posedirs",
        "J_regressor",
        "weights",
        "kintree_table",
        "f",
    ]:
        if key in data.files:
            obj = data[key]
            print(
                f"{key:20s}",
                "shape =", getattr(obj, "shape", None),
                "dtype =", getattr(obj, "dtype", None)
            )

    # Useful sanity checks
    if "v_template" in data.files:
        print(
            "\nVertex count:",
            data["v_template"].shape[0]
        )

    if "shapedirs" in data.files:
        print(
            "Shape components:",
            data["shapedirs"].shape[-1]
        )

    if "expr_dirs" in data.files:
        print(
            "Expression components:",
            data["expr_dirs"].shape[-1]
        )

    data.close()

print("\n" + "=" * 100)
print("DONE")
print("=" * 100)

In [ ]:
from pathlib import Path
import shutil
import numpy as np

SOURCE = Path(
    "/kaggle/input/models/dravvalt/"
    "smplx-neutral-v1-1-private/pytorch/default/1/"
    "models/smplx/SMPLX_NEUTRAL.npz"
)

TARGET = Path(
    "/kaggle/working/multi-hmr/models/smplx/SMPLX_NEUTRAL.npz"
)

TARGET.parent.mkdir(parents=True, exist_ok=True)

assert SOURCE.exists(), f"❌ Source missing: {SOURCE}"

shutil.copy2(SOURCE, TARGET)

assert TARGET.exists()

# Final sanity check
data = np.load(TARGET, allow_pickle=True)

assert data["v_template"].shape == (10475, 3)
assert data["shapedirs"].shape[-1] == 400
assert data["J_regressor"].shape[1] == 10475

data.close()

print("✅ SMPL-X FULL v1.1 MODEL READY")
print("Target:", TARGET)
print("Size MB:", round(TARGET.stat().st_size / 1024**2, 2))
print("SMPLX_ASSET_READY=PASS")

In [ ]:
# ============================================================
# CELL 26 — FIRST REAL MULTI-HMR INFERENCE ON VIDEO_05
# ============================================================

import subprocess
from pathlib import Path

REPO = Path("/kaggle/working/multi-hmr")
PYTHON = Path("/kaggle/working/envs/multihmr_py39/bin/python")

VIDEO = Path(
    "/kaggle/input/datasets/dravvalt/drishti-dataset/"
    "03.CCTV Mobile Usage.mkv"
)

FRAME_DIR = Path("/kaggle/working/drishti_multihmr/frame_test")
OUT_DIR = Path("/kaggle/working/drishti_multihmr/output")

FRAME_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)

SMPLX = REPO / "models" / "smplx" / "SMPLX_NEUTRAL.npz"

print("=" * 100)
print("FIRST REAL MULTI-HMR INFERENCE")
print("=" * 100)

print("Video:", VIDEO)
print("SMPL-X:", SMPLX)
print("SMPL-X exists:", SMPLX.exists())

assert VIDEO.exists(), f"❌ Video missing: {VIDEO}"


# ------------------------------------------------------------
# 1. EXTRACT REPRESENTATIVE FRAME
# ------------------------------------------------------------

print("\n[1] EXTRACTING VIDEO_05 FRAME")
print("-" * 100)

FRAME = FRAME_DIR / "video05_140s.jpg"

cmd = [
    "ffmpeg",
    "-y",
    "-ss", "140.567",
    "-i", str(VIDEO),
    "-frames:v", "1",
    "-q:v", "2",
    str(FRAME),
]

r = subprocess.run(
    cmd,
    capture_output=True,
    text=True
)

assert r.returncode == 0, (
    "❌ Frame extraction failed.\n" + r.stderr[-3000:]
)

assert FRAME.exists()

print("Frame:", FRAME)
print("Size KB:", round(FRAME.stat().st_size / 1024, 2))


# ------------------------------------------------------------
# 2. CHECK SMPL-X GATE
# ------------------------------------------------------------

print("\n[2] SMPL-X ASSET GATE")
print("-" * 100)

if not SMPLX.exists():
    print("❌ SMPLX_NEUTRAL.npz IS MISSING")
    print()
    print("Expected exact path:")
    print(SMPLX)
    print()
    print("Frame extraction succeeded.")
    print("Inference cannot proceed until licensed SMPL-X asset is mounted.")
    
else:
    print("✅ SMPL-X asset present.")


# ------------------------------------------------------------
# 3. RUN REAL MULTI-HMR DEMO IF ASSET EXISTS
# ------------------------------------------------------------

if SMPLX.exists():

    print("\n[3] RUNNING MULTI-HMR INFERENCE")
    print("-" * 100)

    cmd = [
        str(PYTHON),
        "demo.py",

        "--img_folder",
        str(FRAME_DIR),

        "--out_folder",
        str(OUT_DIR),

        "--model_name",
        "multiHMR_672_S",

        "--save_mesh",
        "1",

        "--extra_views",
        "0",

        "--save_rotating_video",
        "0",
    ]

    print("Command:")
    print(" ".join(cmd))

    r = subprocess.run(
        cmd,
        cwd=REPO,
        capture_output=True,
        text=True
    )

    print("\nSTDOUT:")
    print(r.stdout)

    if r.stderr.strip():
        print("\nSTDERR:")
        print(r.stderr)

    assert r.returncode == 0, (
        "❌ Multi-HMR inference failed."
    )

    print("\n✅ REAL MULTI-HMR INFERENCE COMPLETED")


# ------------------------------------------------------------
# 4. INVENTORY OUTPUTS
# ------------------------------------------------------------

print("\n[4] OUTPUT INVENTORY")
print("-" * 100)

if OUT_DIR.exists():

    files = sorted(
        p for p in OUT_DIR.rglob("*")
        if p.is_file()
    )

    if not files:
        print("No output files yet.")
    else:
        for p in files:
            print(
                p.relative_to(OUT_DIR),
                round(p.stat().st_size / 1024, 2),
                "KB"
            )


print("\n" + "=" * 100)
print("CELL 26 COMPLETE")
print("=" * 100)

if not SMPLX.exists():
    print(
        "STATUS: BLOCKED ONLY BY SMPL-X LICENSED ASSET.\n"
        "Everything else is ready."
    )
else:
    print(
        "STATUS: FIRST MULTI-HMR INFERENCE ATTEMPT COMPLETE."
    )